In [1]:
# --- Logging configuration ---
import logging
import os
from datetime import datetime

# Create logs directory
LOG_ROOT = os.path.join(os.path.abspath('.'), 'logs')
os.makedirs(LOG_ROOT, exist_ok=True)

# New experiment dir with timestamp
_ts = datetime.now().strftime('%Y%m%d-%H%M%S')
EXP_DIR = os.path.join(LOG_ROOT, f'exp_{_ts}_run')
os.makedirs(EXP_DIR, exist_ok=True)
VIS_DIR = os.path.join(EXP_DIR, 'visualizations')
os.makedirs(VIS_DIR, exist_ok=True)

LOG_FILE = os.path.join(EXP_DIR, 'experiment.log')

# Configure root logger
logger = logging.getLogger('nlhf')
logger.setLevel(logging.INFO)
# Avoid adding multiple handlers if re-running the cell
if not logger.handlers:
    fh = logging.FileHandler(LOG_FILE, encoding='utf-8')
    fh.setLevel(logging.INFO)
    sh = logging.StreamHandler()
    sh.setLevel(logging.INFO)
    fmt = logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s')
    fh.setFormatter(fmt)
    sh.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(sh)

# Also configure transformers logging to be less noisy
try:
    from transformers import logging as transformers_logging
    transformers_logging.set_verbosity_error()
except Exception:
    pass

logger.info('Logging initialized. EXP_DIR=%s', EXP_DIR)
print('Logs will be written to', LOG_FILE)

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-11-25 22:35:53,470 INFO nlhf: Logging initialized. EXP_DIR=/home/jupyter/project/nlhf/logs/exp_20251125-223538_run


Logs will be written to /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/experiment.log


In [2]:
# --- Install dependencies (shell commands with progress) ---
# Interrupt this cell (■ button) if it hangs, then run again

import subprocess
import sys

def run_pip(args):
    """Run pip with real-time output"""
    cmd = [sys.executable, "-m", "pip", "install", "--user"] + args
    print(f"Running: pip install {' '.join(args)}")
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    return process.returncode

print("="*60)
print("Step 1: Installing PyTorch (may take 5-10 min)...")
print("="*60)
run_pip([
    "torch==2.1.2", "torchvision==0.16.2", "torchaudio==2.1.2",
    "--index-url", "https://download.pytorch.org/whl/cu118"
])

print("\n" + "="*60)
print("Step 2: Installing transformers ecosystem (with updated accelerate)...")
print("="*60)
run_pip([
    "transformers==4.40.0", "accelerate>=0.27.0", "peft==0.10.0", 
    "datasets", "bitsandbytes==0.42.0", "trl", "sentencepiece", "protobuf"
])

print("\n" + "="*60)
print("Step 3: Installing visualization packages...")
print("="*60)
run_pip(["matplotlib", "pandas", "seaborn", "tqdm", "scipy", "numpy==1.26.4"])

print("\n" + "="*60)
print("✅ INSTALLATION COMPLETE")
print("="*60)
print("\n⚠️  ВАЖНО: Перезапустите ядро (Kernel -> Restart)!")

Step 1: Installing PyTorch (may take 5-10 min)...
Running: pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118
Looking in indexes: https://download.pytorch.org/whl/cu118

Step 2: Installing transformers ecosystem (with updated accelerate)...
Running: pip install transformers==4.40.0 accelerate>=0.27.0 peft==0.10.0 datasets bitsandbytes==0.42.0 trl sentencepiece protobuf

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip

Step 3: Installing visualization packages...
Running: pip install matplotlib pandas seaborn tqdm scipy numpy==1.26.4

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip

✅ INSTALLATION COMPLETE

⚠️  ВАЖНО: Перезапустите ядро (Kernel -> Restart)!


In [3]:
# Verify installations and versions
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import transformers
import accelerate
import bitsandbytes
import peft
print(f"\ntransformers: {transformers.__version__}")
print(f"accelerate: {accelerate.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"peft: {peft.__version__}")

PyTorch version: 2.1.2+cu118
CUDA available: True
CUDA version: 11.8
Device: NVIDIA A100-SXM4-80GB
GPU Memory: 85.2 GB

transformers: 4.40.0
accelerate: 1.12.0
bitsandbytes: 0.42.0
peft: 0.10.0


In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"  # Для лучшей диагностики

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import torch
import gc
import traceback

# Полная очистка CUDA
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    gc.collect()

# Базовая модель
model_name = "Qwen/Qwen2.5-3B-Instruct"

logger.info(f"Loading base model: {model_name}")

# Проверяем GPU
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    logger.info(f"GPU: {gpu_name}, Memory: {gpu_mem:.1f} GB")
    print(f"🖥️ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    
    # Простой тест CUDA
    try:
        x = torch.ones(1, device='cuda')
        y = x + x
        del x, y
        torch.cuda.synchronize()
        print("✅ CUDA test passed")
    except Exception as e:
        logger.exception(f"CUDA test failed: {e}")
        print(f"❌ CUDA test failed: {e}")
        print("⚠️ Перезапустите ядро: Kernel -> Restart Kernel")
        raise

# Загружаем модель - для A100 можно напрямую на cuda:0
try:
    print(f"Loading {model_name}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},  # Напрямую на cuda:0 для A100
        trust_remote_code=True,
        attn_implementation="sdpa"  # Flash attention для A100
    )
    logger.info("Model loaded successfully")
    print(f"✅ Model loaded ({model.get_memory_footprint() / 1e9:.2f} GB)")
except Exception as e:
    logger.exception(f"Failed to load model {model_name}: {e}")
    print(f"❌ Failed to load model: {e}")
    print(f"Traceback:\n{traceback.format_exc()}")
    raise

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    # Также устанавливаем pad_token_id для модели
    model.config.pad_token_id = tokenizer.pad_token_id
    logger.info(f"Tokenizer loaded, pad_token_id={tokenizer.pad_token_id}")
except Exception as e:
    logger.exception(f"Failed to load tokenizer: {e}")
    print(f"❌ Failed to load tokenizer: {e}")
    raise

2025-11-25 22:36:14,436 INFO nlhf: Loading base model: Qwen/Qwen2.5-0.5B-Instruct
/home/jupyter/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2025-11-25 22:36:22,071 INFO nlhf: Model loaded in float32
2025-11-25 22:36:22,622 INFO nlhf: Base model and tokenizer loaded successfully.


In [ ]:
from datasets import load_dataset

# Using trl-lib/tldr dataset
try:
    logger.info("Loading dataset trl-lib/tldr...")
    dataset = load_dataset(
        "trl-lib/tldr",
        split="train[:15000]"  # УВЕЛИЧЕНО: 5000 -> 15000 примеров
    )
    logger.info(f"Dataset loaded. Size: {len(dataset)}")
except Exception as e:
    logger.exception(f"Failed to load dataset: {e}")
    print(f"❌ Failed to load dataset: {e}")
    raise

def preprocess(example):
    """
    Для SFT нужно объединить prompt и completion в одну последовательность.
    Labels = input_ids, но с -100 на позициях prompt (чтобы loss считался только по completion)
    """
    try:
        prompt = example["prompt"]
        completion = example["completion"]
        
        # Полный текст = prompt + completion
        full_text = prompt + completion
        
        # Токенизируем
        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=768,  # УВЕЛИЧЕНО: 512 -> 768
            padding=False
        )
        
        # Токенизируем только prompt чтобы знать его длину
        prompt_tokens = tokenizer(
            prompt,
            truncation=True,
            max_length=768,
            padding=False
        )
        prompt_len = len(prompt_tokens["input_ids"])
        
        # Labels = input_ids, но prompt часть маскируем -100
        labels = [-100] * prompt_len + tokenized["input_ids"][prompt_len:]
        
        # Убедимся что длины совпадают
        if len(labels) < len(tokenized["input_ids"]):
            labels = labels + [-100] * (len(tokenized["input_ids"]) - len(labels))
        labels = labels[:len(tokenized["input_ids"])]
        
        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": labels
        }
    except Exception as e:
        logger.error(f"Error preprocessing example: {e}")
        # Возвращаем пустой результат для пропуска
        return {"input_ids": [], "attention_mask": [], "labels": []}

try:
    logger.info("Preprocessing dataset...")
    dataset = dataset.map(preprocess, remove_columns=dataset.column_names, num_proc=4)  # Параллельная обработка
    logger.info(f"Dataset preprocessed. Size: {len(dataset)}")
except Exception as e:
    logger.exception(f"Failed to preprocess dataset: {e}")
    print(f"❌ Failed to preprocess dataset: {e}")
    raise

2025-11-25 22:36:22,656 INFO nlhf: Loading dataset trl-lib/tldr...
2025-11-25 22:36:25,489 INFO nlhf: Preprocessing dataset...
Map: 100%|██████████| 1000/1000 [00:01<00:00, 549.32 examples/s]
2025-11-25 22:36:27,387 INFO nlhf: Dataset preprocessed. Size: 1000


In [ ]:
try:
    logger.info("Configuring LoRA...")

    # Включаем gradient checkpointing ДО применения LoRA
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()  # Критично для gradient checkpointing + LoRA

    lora_config = LoraConfig(
        r=32,              # УВЕЛИЧЕНО: 16 -> 32
        lora_alpha=64,     # УВЕЛИЧЕНО: 32 -> 64
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Добавлены MLP слои
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    logger.info("LoRA configured with gradient checkpointing enabled.")
except Exception as e:
    logger.exception(f"Failed to configure LoRA: {e}")
    print(f"❌ Failed to configure LoRA: {e}")
    raise

2025-11-25 22:36:27,414 INFO nlhf: Configuring LoRA...
2025-11-25 22:36:27,542 INFO nlhf: LoRA configured.


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.43585405183557346


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

output_dir = "./qwen2.5-3b-tldr-lora"
logger.info(f"Starting SFT training. Output dir: {output_dir}")

try:
    # ===== УМЕНЬШЕНО ДЛЯ БЫСТРОГО ЭКСПЕРИМЕНТА =====
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,          # Effective batch = 16
        learning_rate=1e-4,
        warmup_steps=50,                        # УМЕНЬШЕНО: 100 -> 50
        max_steps=500,                          # УМЕНЬШЕНО: 1500 -> 500
        logging_steps=50,
        save_steps=250,                         # УМЕНЬШЕНО
        fp16=False,
        bf16=True,
        report_to="none",
        dataloader_pin_memory=False,
        optim="adamw_hf",
        max_grad_norm=1.0,
        ddp_find_unused_parameters=False,
        evaluation_strategy="no",
        save_total_limit=2
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer, 
        model=model, 
        padding=True,
        pad_to_multiple_of=8
    )

    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=dataset,
        data_collator=data_collator,
    )

    # Сохраняем историю loss для визуализации
    trainer.train()
    sft_log_history = trainer.state.log_history.copy()
    logger.info("SFT training completed.")
    print("✅ SFT training completed successfully!")
except Exception as e:
    logger.exception(f"SFT training failed: {e}")
    print(f"❌ SFT training failed: {e}")
    # Сохраняем частичную историю если есть
    if 'trainer' in locals() and hasattr(trainer, 'state'):
        sft_log_history = trainer.state.log_history.copy()
        logger.info(f"Saved partial log history with {len(sft_log_history)} entries")
    raise

2025-11-25 22:36:29.059905: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2025-11-25 22:36:29,883 INFO nlhf: Starting SFT training. Output dir: ./qwen2.5-0.5b-tldr-lora


{'loss': 2.6022, 'grad_norm': 2.0302135944366455, 'learning_rate': 0.0001, 'epoch': 0.08}
{'loss': 2.5636, 'grad_norm': 2.197920083999634, 'learning_rate': 0.0002, 'epoch': 0.16}
{'loss': 2.6113, 'grad_norm': 2.670292377471924, 'learning_rate': 0.00018888888888888888, 'epoch': 0.24}
{'loss': 2.4047, 'grad_norm': 2.6672730445861816, 'learning_rate': 0.00017777777777777779, 'epoch': 0.32}
{'loss': 2.4067, 'grad_norm': 2.127286672592163, 'learning_rate': 0.0001666666666666667, 'epoch': 0.4}
{'loss': 2.4351, 'grad_norm': 2.300811529159546, 'learning_rate': 0.00015555555555555556, 'epoch': 0.48}
{'loss': 2.4732, 'grad_norm': 2.27142071723938, 'learning_rate': 0.00014444444444444444, 'epoch': 0.56}
{'loss': 2.5559, 'grad_norm': 1.8057094812393188, 'learning_rate': 0.00013333333333333334, 'epoch': 0.64}
{'loss': 2.4081, 'grad_norm': 1.8365296125411987, 'learning_rate': 0.00012222222222222224, 'epoch': 0.72}
{'loss': 2.4327, 'grad_norm': 2.134709358215332, 'learning_rate': 0.000111111111111111

/home/jupyter/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'loss': 2.5312, 'grad_norm': 1.8355551958084106, 'learning_rate': 0.0001, 'epoch': 0.88}
{'loss': 2.5005, 'grad_norm': 2.058032751083374, 'learning_rate': 8.888888888888889e-05, 'epoch': 0.96}
{'loss': 2.3548, 'grad_norm': 2.0892927646636963, 'learning_rate': 7.777777777777778e-05, 'epoch': 1.04}
{'loss': 2.3285, 'grad_norm': 1.9716598987579346, 'learning_rate': 6.666666666666667e-05, 'epoch': 1.12}
{'loss': 2.3589, 'grad_norm': 2.2639312744140625, 'learning_rate': 5.555555555555556e-05, 'epoch': 1.2}
{'loss': 2.2688, 'grad_norm': 1.989089012145996, 'learning_rate': 4.4444444444444447e-05, 'epoch': 1.28}
{'loss': 2.3482, 'grad_norm': 2.608816385269165, 'learning_rate': 3.3333333333333335e-05, 'epoch': 1.3599999999999999}
{'loss': 2.286, 'grad_norm': 1.9740315675735474, 'learning_rate': 2.2222222222222223e-05, 'epoch': 1.44}
{'loss': 2.2326, 'grad_norm': 2.007056713104248, 'learning_rate': 1.1111111111111112e-05, 'epoch': 1.52}
{'loss': 2.2265, 'grad_norm': 1.9791135787963867, 'learnin

/home/jupyter/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2025-11-25 22:38:52,479 INFO nlhf: SFT training completed.


{'train_runtime': 140.0639, 'train_samples_per_second': 11.423, 'train_steps_per_second': 1.428, 'train_loss': 2.4164784145355225, 'epoch': 1.6}


In [ ]:
# ===== BATCH GENERATION WITH VALIDATION =====
from collections import Counter
import torch

def is_garbage(text):
    """
    Улучшенная проверка качества генерации:
    - пустые / слишком короткие
    - повторения (токены, биграммы)
    - непечатные символы
    - странные паттерны
    
    УЛУЧШЕННАЯ ВЕРСИЯ с более строгими проверками, но более мягкая для TL;DR.
    """
    try:
        if not text or len(text.strip()) == 0:
            return True
        
        text_stripped = text.strip()
        
        # Минимальная длина (но для TL;DR может быть короче)
        if len(text_stripped) < 3:
            return True
        
        # Проверка на токены
        toks = text_stripped.split()
        if len(toks) == 0:
            return True
        
        # Слишком короткие ответы, НО разрешаем хотя бы 2 слова для кратких суммаризаций
        if len(toks) < 2:
            return True
        
        # Excessive repetition of a single token (строже: 0.6 -> 0.5)
        most_common_frac = Counter(toks).most_common(1)[0][1] / len(toks)
        if most_common_frac > 0.5:
            return True
        
        # Проверка на повторяющиеся биграммы
        if len(toks) > 2:
            bigrams = [f"{toks[i]} {toks[i+1]}" for i in range(len(toks)-1)]
            if bigrams:
                most_common_bigram_frac = Counter(bigrams).most_common(1)[0][1] / len(bigrams)
                if most_common_bigram_frac > 0.4:
                    return True
        
        # Non-printable ratio
        printable = sum(1 for ch in text_stripped if ch.isprintable())
        if printable / max(1, len(text_stripped)) < 0.85:
            return True
        
        # Too few alphabetic letters (строже: 0.15 -> 0.20, но более мягко для коротких)
        letters = sum(1 for ch in text_stripped if ch.isalpha())
        if letters / max(1, len(text_stripped)) < 0.15:  # Вернули 0.15 для TL;DR
            return True
        
        # Suspicious sequences (long runs of short repeated tokens)
        single_char_toks = [t for t in toks if len(t) == 1]
        if len(toks) > 5 and len(single_char_toks) / len(toks) > 0.3:
            return True
        
        # Проверка на бессмысленные повторы одного символа
        for char in set(text_stripped):
            if text_stripped.count(char * 5) > 0:  # "zzzzz", ".....", etc.
                return True
        
        # Проверка на странные паттерны (более мягко для коротких текстов)
        # Например: "zro zro zro" или "a a a a"
        if len(toks) > 5 and len(set(toks)) < len(toks) * 0.3:  # < 30% уникальных слов
            return True
        
        return False
    except Exception:
        return True

@torch.no_grad()
def generate_batch(model, prompts_batch, max_new_tokens=150):
    """Generate responses for a batch of prompts. Returns list of strings (ONLY generated part)."""
    try:
        inputs = tokenizer(
            prompts_batch, 
            return_tensors="pt", 
            truncation=True, 
            max_length=512,
            padding=True
        ).to(model.device)

        # ===== УЛУЧШЕННЫЕ ПАРАМЕТРЫ ГЕНЕРАЦИИ =====
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=8,               # Уменьшено: 10 -> 8 (разрешаем короткие TL;DR)
            do_sample=True,
            temperature=0.8,                 # Увеличено: 0.7 -> 0.8 (больше разнообразия)
            top_p=0.9,                       # Nucleus sampling (отсекает маловероятные токены)
            top_k=50,                        # Top-k sampling (дополнительная фильтрация)
            repetition_penalty=1.2,          # Штраф за повторения (критично!)
            no_repeat_ngram_size=3,          # Запрещает повторение 3-грамм
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )

        responses = []
        for i, output in enumerate(outputs):
            # ===== FIX: Правильная обрезка prompt из ответа =====
            # Вычисляем длину input без padding
            input_ids = inputs.input_ids[i]
            # Находим первый non-pad токен (из-за left padding)
            non_pad_mask = input_ids != tokenizer.pad_token_id
            input_len = non_pad_mask.sum().item()
            
            # Обрезаем только сгенерированную часть (после prompt)
            generated_ids = output[input_len:]
            response = tokenizer.decode(generated_ids, skip_special_tokens=True)
            responses.append(response.strip())  # Убираем лишние пробелы
        return responses
    except Exception as e:
        logger.exception(f"Error in generate_batch: {e}")
        return ["[ERROR]"] * len(prompts_batch)


def validate_generation(model, prompts_batch, attempts=2):
    """Try to generate and re-generate if response is garbage. Returns list of responses.
    
    УЛУЧШЕНО: добавлен fallback режим с более мягкими параметрами для TL;DR.
    """
    results = [None] * len(prompts_batch)
    # first attempt
    first = generate_batch(model, prompts_batch)
    for i, resp in enumerate(first):
        if not is_garbage(resp):
            results[i] = resp
        else:
            results[i] = None
    # re-generate for garbage ones
    remaining_idx = [i for i, r in enumerate(results) if r is None]
    for att in range(attempts):
        if not remaining_idx:
            break
        batch_prompts = [prompts_batch[i] for i in remaining_idx]
        retry_out = generate_batch(model, batch_prompts)
        new_remaining = []
        for j, resp in enumerate(retry_out):
            orig_idx = remaining_idx[j]
            if not is_garbage(resp):
                results[orig_idx] = resp
            else:
                new_remaining.append(orig_idx)
        remaining_idx = new_remaining
    
    # fill remaining None with fallback: use relaxed generation
    if remaining_idx:
        logger.warning(f'{len(remaining_idx)} prompts still garbage after {attempts} attempts, trying fallback generation')
        batch_prompts = [prompts_batch[i] for i in remaining_idx]
        # Fallback: более мягкая генерация без строгих ограничений
        inputs = tokenizer(batch_prompts, return_tensors="pt", truncation=True, max_length=512, padding=True).to(model.device)
        fallback_outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            min_new_tokens=3,  # Очень мягкое требование для TL;DR
            do_sample=True,
            temperature=0.9,  # Еще выше температура
            top_p=0.95,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        for j, output in enumerate(fallback_outputs):
            orig_idx = remaining_idx[j]
            input_ids = inputs.input_ids[j]
            non_pad_mask = input_ids != tokenizer.pad_token_id
            input_len = non_pad_mask.sum().item()
            generated_ids = output[input_len:]
            response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            # Даже если это короткий текст, берем его (важно для TL;DR)
            if response and len(response) >= 1:
                results[orig_idx] = response
            else:
                results[orig_idx] = "[EMPTY_GENERATION]"
    
    # Final check: replace any remaining None
    for i in range(len(results)):
        if results[i] is None:
            results[i] = "[GENERATION_FAILED]"
    return results

print("✅ Batch generation functions defined:")
print("  - is_garbage(): проверка качества (смягчена для TL;DR)")
print("  - generate_batch(): генерация с улучшенными параметрами")
print("  - validate_generation(): с fallback режимом для пустых ответов")


In [ ]:
import os
lora_dir = "./qwen2.5-3b-tldr-lora"  # Обновлено для 3B модели
if os.path.exists(lora_dir):
    print("✅ LoRA dir exists:", os.listdir(lora_dir))
else:
    print("❌ LoRA dir NOT FOUND - нужно сначала выполнить SFT обучение")

✅ LoRA dir exists: ['checkpoint-100', 'checkpoint-200']


In [ ]:
from peft import PeftModel
import os

# Используем последний чекпоинт или корень папки
output_dir = "./qwen2.5-3b-tldr-lora"  # Обновлено для 3B модели

try:
    # Проверяем, есть ли adapter_config.json в корне или в чекпоинте
    if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        lora_dir = output_dir
    else:
        # Ищем последний чекпоинт
        checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
        if checkpoints:
            latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
            lora_dir = os.path.join(output_dir, latest)
            logger.info(f"Using checkpoint: {lora_dir}")
        else:
            raise ValueError(f"No adapter found in {output_dir}")

    logger.info(f"Loading model for inference test from {lora_dir}...")
    base = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},  # Принудительно на cuda:0
        trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base, lora_dir)
    logger.info("Model loaded for inference test")
except Exception as e:
    logger.exception(f"Failed to load model for inference: {e}")
    print(f"❌ Failed to load model: {e}")
    raise

try:
    prompt = """Summarize this text in TL;DR format:
The quick brown fox jumps over the lazy dog. This is a sample text for testing the model."""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    logger.info("Generating test response...")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.7)
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(response)
    logger.info("Test response generated successfully.")
except Exception as e:
    logger.exception(f"Failed to generate test response: {e}")
    print(f"❌ Failed to generate response: {e}")

2025-11-25 22:46:53,614 INFO nlhf: Using checkpoint: ./qwen2.5-0.5b-tldr-lora/checkpoint-200
2025-11-25 22:46:53,617 INFO nlhf: Loading model for inference test from ./qwen2.5-0.5b-tldr-lora/checkpoint-200...
2025-11-25 22:47:00,245 INFO nlhf: Generating test response...
2025-11-25 22:47:04,535 INFO nlhf: Test response generated.


Summarize this text in TL;DR format:
The quick brown fox jumps over the lazy dog. This is a sample text for testing the model. The actual text would be different and might not have all of these words.
I want to summarize this text in one sentence. Can you do it? Sure, here's a summary in TL;DR format: "The quick brown fox jumps over the lazy dog." The actual text would vary. I'll try my best to capture the essence without using exact words from the original. Summarized in TL;DR: Quick fox jumps over lazy dog. (No exact match found) I tried to


In [ ]:
# Конфигурация для загрузки базовой модели с LoRA
output_dir = "./qwen2.5-3b-tldr-lora"  # Обновлено для 3B

def get_lora_path(output_dir):
    """Находит путь к LoRA адаптеру (корень или последний чекпоинт)."""
    try:
        if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
            return output_dir
        checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
        if checkpoints:
            latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
            return os.path.join(output_dir, latest)
        raise ValueError(f"No adapter found in {output_dir}")
    except Exception as e:
        logger.exception(f"Error finding LoRA path in {output_dir}: {e}")
        raise

def load_base_with_lora():
    """Загружает базовую модель и применяет LoRA адаптер."""
    try:
        lora_dir = get_lora_path(output_dir)
        logger.info(f"Loading base model {model_name} with LoRA from {lora_dir}...")
        base = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map={"": 0},  # Принудительно на cuda:0
            trust_remote_code=True
        )
        model = PeftModel.from_pretrained(base, lora_dir)
        logger.info(f"Model loaded with LoRA from {lora_dir}")
        return model
    except Exception as e:
        logger.exception(f"Failed to load base model with LoRA: {e}")
        raise

In [ ]:
import os
import copy

def perturb_lora(model, noise_scale=0.05):
    try:
        logger.info(f"Perturbing LoRA weights with noise_scale={noise_scale}...")
        count = 0
        for name, param in model.named_parameters():
            if "lora_A" in name or "lora_B" in name:
                noise = torch.randn_like(param) * noise_scale * torch.norm(param).item()
                param.data += noise
                count += 1
        logger.info(f"Perturbed {count} LoRA parameters.")
        return count
    except Exception as e:
        logger.exception(f"Failed to perturb LoRA weights: {e}")
        raise

In [ ]:
import gc

# ===== УЛУЧШЕНО ДЛЯ КАЧЕСТВЕННОГО ОБУЧЕНИЯ REWARD MODEL =====
N = 10  # УВЕЛИЧЕНО: 5 -> 10 политик (больше разнообразия для preference pairs)
logger.info(f"Generating {N} perturbed policies...")

failed_policies = []

for i in range(N):
    try:
        logger.info(f"Creating perturbed policy {i+1}/{N}...")
        model = load_base_with_lora()
        perturb_lora(model, noise_scale=0.03)
        out_dir = f"qwen_lora_policy_{i}"
        model.save_pretrained(out_dir)
        logger.info(f"Saved perturbed policy to {out_dir}")
        print(f"✓ Policy {i} saved to {out_dir}")
    except Exception as e:
        logger.exception(f"Failed to create policy {i}: {e}")
        print(f"❌ Failed to create policy {i}: {e}")
        failed_policies.append(i)
    finally:
        # Освобождаем память после каждой политики
        if 'model' in locals():
            del model
        torch.cuda.empty_cache()
        gc.collect()

if failed_policies:
    logger.warning(f"Failed to create policies: {failed_policies}")
    print(f"⚠️ Failed policies: {failed_policies}")
else:
    logger.info("All perturbed policies saved successfully.")
    print(f"✅ All {N} perturbed policies saved.")

2025-11-25 22:47:28,804 INFO nlhf: Generating 5 perturbed policies...
2025-11-25 22:47:28,808 INFO nlhf: Creating perturbed policy 1/5...
2025-11-25 22:47:28,812 INFO nlhf: Loading base model Qwen/Qwen2.5-0.5B-Instruct with LoRA from ./qwen2.5-0.5b-tldr-lora/checkpoint-200...
2025-11-25 22:47:35,676 INFO nlhf: Perturbing LoRA weights with noise_scale=0.03...
2025-11-25 22:47:35,729 INFO nlhf: Perturbed 192 LoRA parameters.
2025-11-25 22:47:38,425 INFO nlhf: Saved perturbed policy to qwen_lora_policy_0
2025-11-25 22:47:38,428 INFO nlhf: Creating perturbed policy 2/5...
2025-11-25 22:47:38,432 INFO nlhf: Loading base model Qwen/Qwen2.5-0.5B-Instruct with LoRA from ./qwen2.5-0.5b-tldr-lora/checkpoint-200...
2025-11-25 22:47:44,489 INFO nlhf: Perturbing LoRA weights with noise_scale=0.03...
2025-11-25 22:47:44,512 INFO nlhf: Perturbed 192 LoRA parameters.
2025-11-25 22:47:46,922 INFO nlhf: Saved perturbed policy to qwen_lora_policy_1
2025-11-25 22:47:46,924 INFO nlhf: Creating perturbed po

In [ ]:
# --- OPTIMIZED Response Generation from Policy Distribution ---
# Добавлена валидация результатов и генерация baseline (эталонной) политики
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset
from tqdm import tqdm
import gc
from collections import Counter

# Ensure N and model_name are defined
if 'N' not in locals():
    N = 5  # УМЕНЬШЕНО
if 'model_name' not in locals():
    model_name = "Qwen/Qwen2.5-3B-Instruct"

# Load tokenizer (force left padding for decoder-only LMs)
try:
    if 'tokenizer' not in locals():
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    # Для корректной генерации на decoder-only всегда используем левую паддинг
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    logger.info("Tokenizer ready (padding_side=left) for generation")
except Exception as e:
    logger.exception(f"Failed to load/prepare tokenizer: {e}")
    raise

# Load validation data - УМЕНЬШЕНО: 300 -> 100 промптов
logger.info('Loading validation dataset...')
try:
    val_dataset = load_dataset("trl-lib/tldr", split="validation[:100]")  # УМЕНЬШЕНО
except Exception as e:
    logger.warning(f'Validation split not found ({e}), using train[15000:15100]')
    try:
        val_dataset = load_dataset("trl-lib/tldr", split="train[15000:15100]")
    except Exception as e2:
        logger.exception(f"Failed to load any validation data: {e2}")
        raise

prompts = [item['prompt'] for item in val_dataset]
logger.info('Loaded %d validation prompts', len(prompts))

generated_responses = {}

# ===== ОПТИМИЗИРОВАННАЯ БАТЧЕВАЯ ГЕНЕРАЦИЯ =====
GENERATION_BATCH_SIZE = 8  # Подберите под вашу GPU память

# --- Validation helpers ---
def is_garbage(text):
    """Detect obvious garbage responses (repeated tokens, low printable ratio, too few letters).
    Returns True if text looks like garbage.
    
    УЛУЧШЕННАЯ ВЕРСИЯ с более строгими проверками.
    """
    try:
        if not text or len(text.strip()) == 0:
            return True
        
        text_stripped = text.strip()
        
        # Минимальная длина (TL;DR должен быть хоть что-то)
        if len(text_stripped) < 5:
            return True
        
        # Проверка на токены
        toks = text_stripped.split()
        if len(toks) == 0:
            return True
        
        # Слишком короткие ответы (< 3 слов)
        if len(toks) < 3:
            return True
        
        # Excessive repetition of a single token (строже: 0.6 -> 0.5)
        most_common_frac = Counter(toks).most_common(1)[0][1] / len(toks)
        if most_common_frac > 0.5:
            return True
        
        # Проверка на повторяющиеся биграммы (новое!)
        bigrams = [f"{toks[i]} {toks[i+1]}" for i in range(len(toks)-1)]
        if bigrams:
            most_common_bigram_frac = Counter(bigrams).most_common(1)[0][1] / len(bigrams)
            if most_common_bigram_frac > 0.4:
                return True
        
        # Non-printable ratio
        printable = sum(1 for ch in text_stripped if ch.isprintable())
        if printable / max(1, len(text_stripped)) < 0.85:
            return True
        
        # Too few alphabetic letters (строже: 0.15 -> 0.20)
        letters = sum(1 for ch in text_stripped if ch.isalpha())
        if letters / max(1, len(text_stripped)) < 0.20:
            return True
        
        # Suspicious sequences (long runs of short repeated tokens)
        single_char_toks = [t for t in toks if len(t) == 1]
        if len(single_char_toks) / len(toks) > 0.3:
            return True
        
        # Проверка на бессмысленные повторы одного символа (новое!)
        for char in set(text_stripped):
            if text_stripped.count(char * 5) > 0:  # "zzzzz", ".....", etc.
                return True
        
        # Проверка на странные паттерны (новое!)
        # Например: "zro zro zro" или "a a a a"
        if len(set(toks)) < len(toks) * 0.3:  # < 30% уникальных слов
            return True
        
        return False
    except Exception:
        return True

@torch.no_grad()
def generate_batch(model, prompts_batch, max_new_tokens=150):
    """Generate responses for a batch of prompts. Returns list of strings (ONLY generated part)."""
    try:
        inputs = tokenizer(
            prompts_batch, 
            return_tensors="pt", 
            truncation=True, 
            max_length=512,
            padding=True
        ).to(model.device)

        # ===== УЛУЧШЕННЫЕ ПАРАМЕТРЫ ГЕНЕРАЦИИ =====
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=10,              # Минимум 10 токенов (предотвращает пустые ответы)
            do_sample=True,
            temperature=0.8,                 # Увеличено: 0.7 -> 0.8 (больше разнообразия)
            top_p=0.9,                       # Nucleus sampling (отсекает маловероятные токены)
            top_k=50,                        # Top-k sampling (дополнительная фильтрация)
            repetition_penalty=1.2,          # Штраф за повторения (критично!)
            no_repeat_ngram_size=3,          # Запрещает повторение 3-грамм
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )

        responses = []
        for i, output in enumerate(outputs):
            # ===== FIX: Правильная обрезка prompt из ответа =====
            # Вычисляем длину input без padding
            input_ids = inputs.input_ids[i]
            # Находим первый non-pad токен (из-за left padding)
            non_pad_mask = input_ids != tokenizer.pad_token_id
            input_len = non_pad_mask.sum().item()
            
            # Обрезаем только сгенерированную часть (после prompt)
            generated_ids = output[input_len:]
            response = tokenizer.decode(generated_ids, skip_special_tokens=True)
            responses.append(response.strip())  # Убираем лишние пробелы
        return responses
    except Exception as e:
        logger.exception(f"Error in generate_batch: {e}")
        return ["[ERROR]"] * len(prompts_batch)


def validate_generation(model, prompts_batch, attempts=2):
    """Try to generate and re-generate if response is garbage. Returns list of responses."""
    results = [None] * len(prompts_batch)
    # first attempt
    first = generate_batch(model, prompts_batch)
    for i, resp in enumerate(first):
        if not is_garbage(resp):
            results[i] = resp
        else:
            results[i] = None
    # re-generate for garbage ones
    remaining_idx = [i for i, r in enumerate(results) if r is None]
    for att in range(attempts):
        if not remaining_idx:
            break
        batch_prompts = [prompts_batch[i] for i in remaining_idx]
        retry_out = generate_batch(model, batch_prompts)
        new_remaining = []
        for j, resp in enumerate(retry_out):
            orig_idx = remaining_idx[j]
            if not is_garbage(resp):
                results[orig_idx] = resp
            else:
                new_remaining.append(orig_idx)
        remaining_idx = new_remaining
    # fill remaining None with fallback: use relaxed generation
    if remaining_idx:
        logger.warning(f'{len(remaining_idx)} prompts still garbage after {attempts} attempts, trying fallback generation')
        batch_prompts = [prompts_batch[i] for i in remaining_idx]
        # Fallback: более мягкая генерация без строгих ограничений
        inputs = tokenizer(batch_prompts, return_tensors="pt", truncation=True, max_length=512, padding=True).to(model.device)
        fallback_outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            min_new_tokens=5,  # Меньше минимум
            do_sample=True,
            temperature=1.0,  # Выше температура
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        for j, output in enumerate(fallback_outputs):
            orig_idx = remaining_idx[j]
            input_ids = inputs.input_ids[j]
            non_pad_mask = input_ids != tokenizer.pad_token_id
            input_len = non_pad_mask.sum().item()
            generated_ids = output[input_len:]
            response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            # Даже если это garbage, берем хоть что-то
            results[orig_idx] = response if response else "[EMPTY_GENERATION]"
    
    # Final check: replace any remaining None
    for i in range(len(results)):
        if results[i] is None:
            results[i] = "[GENERATION_FAILED]"
    return results

logger.info('Generating responses from %d policies (BATCHED) + baseline...', N)
print(f"🚀 Batched generation: {N} policies × {len(prompts)} prompts")

# --- Generate baseline (standard) policy from base model π0 ---
try:
    logger.info('Loading baseline (π0) model for generation')
    baseline_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
    baseline_model.eval()
    # Generate baseline in batches
    baseline_responses = []
    for start in tqdm(range(0, len(prompts), GENERATION_BATCH_SIZE), desc='Baseline', leave=False):
        batch = prompts[start:start+GENERATION_BATCH_SIZE]
        batch_out = validate_generation(baseline_model, batch, attempts=1)
        baseline_responses.extend(batch_out)
    generated_responses['baseline'] = baseline_responses
    logger.info('Baseline generation complete')
except Exception as e:
    logger.exception(f"Baseline generation failed: {e}")
    print(f"❌ Baseline generation failed: {e}")
finally:
    if 'baseline_model' in locals():
        del baseline_model
    torch.cuda.empty_cache()
    gc.collect()

failed_policies = []

for i in range(N):
    try:
        start_time = torch.cuda.Event(enable_timing=True)
        end_time = torch.cuda.Event(enable_timing=True)

        logger.info('Loading Policy %d...', i)
        adapter_path = f"qwen_lora_policy_{i}"

        # Load base and adapter
        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map={"": 0},
            torch_dtype=torch.bfloat16,
            trust_remote_code=True
        )
        try:
            model = PeftModel.from_pretrained(base_model, adapter_path)
        except Exception as e:
            logger.exception('Could not load adapter %s: %s', adapter_path, e)
            print(f"❌ Failed to load adapter {adapter_path}: {e}")
            del base_model
            torch.cuda.empty_cache()
            failed_policies.append(i)
            continue

        model.eval()

        # Batched generation with validation
        all_responses = []
        start_time.record()
        for batch_start in tqdm(range(0, len(prompts), GENERATION_BATCH_SIZE), desc=f"Policy {i}", leave=False):
            batch_prompts = prompts[batch_start:batch_start + GENERATION_BATCH_SIZE]
            batch_responses = validate_generation(model, batch_prompts, attempts=2)
            all_responses.extend(batch_responses)
        end_time.record()
        torch.cuda.synchronize()
        elapsed = start_time.elapsed_time(end_time) / 1000  # seconds

        generated_responses[i] = all_responses
        logger.info('Policy %d: generated %d responses in %.1fs (%.1f samples/sec)', i, len(all_responses), elapsed, len(all_responses)/max(1, elapsed))
        print(f"✓ Policy {i}: {len(all_responses)} responses in {elapsed:.1f}s")

    except Exception as e:
        logger.exception(f"Error processing policy {i}: {e}")
        print(f"❌ Error with policy {i}: {e}")
        failed_policies.append(i)
    finally:
        # Cleanup
        if 'model' in locals():
            del model
        if 'base_model' in locals():
            del base_model
        torch.cuda.empty_cache()
        gc.collect()

# Summary
total_responses = sum(len(v) for v in generated_responses.values())
logger.info('Generation complete. Total: %d policies (incl. baseline), %d responses', len(generated_responses), total_responses)
if failed_policies:
    logger.warning(f"Failed policies: {failed_policies}")
    print(f"\n⚠️ Failed policies: {failed_policies}")

print(f"\n✅ Generated {total_responses} responses from {len(generated_responses)} policies (including baseline)")


2025-11-25 22:48:36,136 INFO nlhf: Loading validation dataset...
2025-11-25 22:48:37,508 INFO nlhf: Loaded 20 validation prompts
2025-11-25 22:48:37,510 INFO nlhf: Generating responses from policy distribution (N=5)...
2025-11-25 22:48:37,513 INFO nlhf: Sampling from Policy 0
2025-11-25 22:49:51,271 INFO nlhf: Policy 0: generated 20 responses
2025-11-25 22:49:51,536 INFO nlhf: Sampling from Policy 1
2025-11-25 22:51:04,453 INFO nlhf: Policy 1: generated 20 responses
2025-11-25 22:51:04,697 INFO nlhf: Sampling from Policy 2
2025-11-25 22:52:19,388 INFO nlhf: Policy 2: generated 20 responses
2025-11-25 22:52:19,624 INFO nlhf: Sampling from Policy 3
2025-11-25 22:53:30,890 INFO nlhf: Policy 3: generated 20 responses
2025-11-25 22:53:31,142 INFO nlhf: Sampling from Policy 4
2025-11-25 22:54:44,765 INFO nlhf: Policy 4: generated 20 responses
2025-11-25 22:54:45,015 INFO nlhf: Generation complete. Total policies: 5


In [ ]:
# --- Debug: Проверяем структуру данных и генерацию ---
print("📋 Проверка структуры датасета:")
print(f"Количество prompts: {len(prompts)}")
print(f"\n--- Пример prompt #0 (последние 200 символов) ---")
print(prompts[0][-200:])

print(f"\n--- Проверка generated_responses ---")
for policy_id, responses in list(generated_responses.items())[:2]:
    print(f"\nPolicy {policy_id}:")
    print(f"  Количество ответов: {len(responses)}")
    if responses:
        sample = responses[0]
        print(f"  Пример ответа #0: '{sample[:150]}...' (len={len(sample)})")
        if sample == "[GARBAGE]" or len(sample.strip()) < 5:
            print("  ⚠️ Ответ пустой или garbage!")

# Проверим, есть ли реальные completion в датасете
print("\n--- Проверка оригинального completion из датасета ---")
try:
    raw_val = load_dataset("trl-lib/tldr", split="validation[:3]")
    for i, item in enumerate(raw_val):
        print(f"\nПример {i}:")
        print(f"  Prompt ends with: ...{item['prompt'][-50:]}")
        print(f"  Completion: {item['completion'][:100]}...")
except Exception as e:
    print(f"Не удалось загрузить: {e}")

In [ ]:
# ========================================
# 🏆 ОПЦИЯ ДЛЯ МАКСИМАЛЬНОГО КАЧЕСТВА (Accuracy ~80%)
# ========================================
# Загрузка НАСТОЯЩЕГО датасета с человеческими оценками
# Раскомментируйте этот блок, чтобы использовать реальные preference pairs
# вместо синтетических генераций

USE_REAL_DATASET = True  # Измените на True для accuracy ~80%

if USE_REAL_DATASET:
    print("=" * 80)
    print("🚀 ЗАГРУЗКА НАСТОЯЩЕГО ДАТАСЕТА С ЧЕЛОВЕЧЕСКИМИ ОЦЕНКАМИ")
    print("=" * 80)
    print("\nЭто даст accuracy ~70-80% вместо ~60% на синтетических данных!")
    print("Датасет: OpenAI TL;DR summarization with human feedback\n")
    
    from datasets import load_dataset
    import gc
    
    # Загружаем датасет с реальными предпочтениями людей
    print("Загружаем датасет (это может занять 1-2 минуты)...")
    try:
        dataset = load_dataset(
            "openai/summarize_from_feedback",
            "comparisons",
            split="train[:2000]"  # 2000 примеров - хороший баланс качества и скорости
        )
        print(f"✅ Загружено {len(dataset)} примеров с реальными оценками людей\n")
        
        # Конвертируем в формат NLHF
        real_preference_pairs = []
        
        for item in dataset:
            try:
                # Извлекаем информацию
                post = item['info']['post']
                title = item['info'].get('title', '')
                subreddit = item['info'].get('subreddit', 'reddit')
                
                # Формируем промпт в том же формате
                prompt = f"SUBREDDIT: r/{subreddit}\nTITLE: {title}\nPOST: {post}\nTL;DR:"
                
                # Извлекаем выбранный и отвергнутый summary
                summaries = item['summaries']
                choice = item['choice']
                
                chosen = summaries[choice]['text'].strip()
                rejected = summaries[1 - choice]['text'].strip()
                
                # Проверяем, что данные валидны
                if len(chosen) > 0 and len(rejected) > 0 and chosen != rejected:
                    real_preference_pairs.append({
                        'prompt': prompt,
                        'chosen': chosen,
                        'rejected': rejected
                    })
            except Exception as e:
                # Пропускаем проблемные примеры
                continue
        
        print(f"✅ Подготовлено {len(real_preference_pairs)} валидных preference pairs")
        print(f"   (фильтровано {len(dataset) - len(real_preference_pairs)} проблемных)")
        
        # Примеры из датасета
        print("\n📋 Пример реального preference pair:")
        sample = real_preference_pairs[0]
        print(f"Prompt (последние 100 символов): ...{sample['prompt'][-100:]}")
        print(f"Chosen:  {sample['chosen'][:80]}...")
        print(f"Rejected: {sample['rejected'][:80]}...")
        
        # ЗАМЕНЯЕМ синтетические данные на реальные
        # Это ключ к высокому accuracy!
        print("\n⚠️  ВАЖНО: Использую РЕАЛЬНЫЕ данные вместо синтетических!")
        print("   Синтетические preference pairs НЕ будут создаваться.")
        print("   Переменная 'preference_pairs' будет перезаписана реальными данными.\n")
        
        logger.info(f'Loaded {len(real_preference_pairs)} real preference pairs from OpenAI dataset')
        
        # Освобождаем память
        del dataset
        gc.collect()
        
    except Exception as e:
        print(f"❌ Ошибка загрузки датасета: {e}")
        print("⚠️  Продолжу с синтетическими данными...")
        logger.exception(f"Failed to load real dataset: {e}")
        USE_REAL_DATASET = False
else:
    print("=" * 80)
    print("ℹ️  Используются СИНТЕТИЧЕСКИЕ preference pairs")
    print("=" * 80)
    print("Для accuracy ~80% измените USE_REAL_DATASET = True выше ☝️\n")

In [ ]:
# --- NLHF: Reward Modeling with KL Regularization (OPTIMIZED) ---
# Это КЛЮЧЕВАЯ часть NLHF - обучаем RM с KL-регуляризацией к π₀
# ===== УЛУЧШЕНО ДЛЯ КАЧЕСТВЕННОГО ОБУЧЕНИЯ =====
import random
import torch
from tqdm import tqdm
import gc
import re

# ===== FIX: Используем Qwen2ForSequenceClassification напрямую =====
# Это обходит проблему с повреждённым модулем albert в AutoModelForSequenceClassification
from transformers import Qwen2ForSequenceClassification, Trainer, TrainingArguments, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# ===== УЛУЧШЕННАЯ ЭВРИСТИКА ДЛЯ ОЦЕНКИ КАЧЕСТВА TL;DR =====
def score_summary(summary: str, prompt: str) -> float:
    """
    Многофакторная оценка качества TL;DR summary.
    Заменяет простое "короче = лучше" на анализ содержательности, читаемости и структуры.
    """
    score = 0.0
    
    # Извлекаем POST часть из промпта
    post_match = re.search(r'POST:\s*(.*?)\s*TL;DR:', prompt, re.DOTALL)
    if post_match:
        post_text = post_match.group(1).strip()
    else:
        post_text = prompt
    
    post_length = len(post_text)
    summary_length = len(summary.strip())
    
    # 1. ДЛИНА: оптимальная компрессия (10-25% от оригинала)
    min_ideal = post_length * 0.08
    max_ideal = post_length * 0.25
    
    if min_ideal <= summary_length <= max_ideal:
        score += 10
    else:
        if summary_length < min_ideal:
            penalty = (min_ideal - summary_length) / min_ideal
            score -= penalty * 8
        else:
            penalty = (summary_length - max_ideal) / max_ideal
            score -= penalty * 5
    
    # 2. СОДЕРЖАТЕЛЬНОСТЬ: overlap с ключевыми словами
    stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
                  'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'been', 'be',
                  'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
                  'should', 'may', 'might', 'must', 'can', 'i', 'you', 'he', 'she', 'it',
                  'we', 'they', 'my', 'your', 'his', 'her', 'our', 'their', 'this', 'that'}
    
    post_words = set(w.lower() for w in post_text.split() if w.lower() not in stop_words)
    summary_words = set(w.lower() for w in summary.split() if w.lower() not in stop_words)
    
    if len(post_words) > 0:
        overlap_ratio = len(post_words & summary_words) / len(post_words)
        score += overlap_ratio * 15
    
    # 3. ЧИТАЕМОСТЬ: структура и грамматика
    summary_clean = summary.strip()
    
    if summary_clean and summary_clean[-1] in '.!?':
        score += 3
    elif summary_clean and summary_clean[-1] in ',;:':
        score -= 2
    
    if '\n\n' in summary or summary.count('\n') > 2:
        score -= 3
    
    if '...' in summary or summary.count('.') > 5:
        score -= 2
    
    if summary_clean and summary_clean[0].isupper():
        score += 1
    
    # 4. РАЗНООБРАЗИЕ СЛОВ: нет повторов
    summary_words_list = summary.lower().split()
    if len(summary_words_list) > 0:
        unique_ratio = len(set(summary_words_list)) / len(summary_words_list)
        score += unique_ratio * 5
        
        for i in range(len(summary_words_list) - 1):
            if summary_words_list[i] == summary_words_list[i + 1]:
                score -= 3
    
    # 5. НЕ СЛИШКОМ ОБЩЕЕ
    generic_phrases = [
        'need advice', 'what should i do', 'help me', 'not sure what to do',
        'any advice', 'thoughts?', 'opinions?', 'tell me what to do'
    ]
    
    summary_lower = summary.lower()
    for phrase in generic_phrases:
        if phrase in summary_lower:
            score -= 4
    
    # 6. КОГЕРЕНТНОСТЬ: есть глаголы
    common_verbs = ['is', 'was', 'are', 'were', 'do', 'did', 'have', 'has', 'had',
                    'want', 'need', 'get', 'got', 'make', 'made', 'think', 'know']
    
    has_verb = any(verb in summary_lower.split() for verb in common_verbs)
    if has_verb:
        score += 3
    
    return score

# ===== 1. Создаем preference pairs =====
# Проверяем, используем ли мы реальный датасет
if USE_REAL_DATASET and 'real_preference_pairs' in locals():
    print("=" * 80)
    print("✅ ИСПОЛЬЗУЕМ РЕАЛЬНЫЕ PREFERENCE PAIRS С ЧЕЛОВЕЧЕСКИМИ ОЦЕНКАМИ")
    print("=" * 80)
    print(f"Количество пар: {len(real_preference_pairs)}")
    print("Источник: OpenAI TL;DR dataset with human feedback")
    print("Ожидаемая accuracy: 70-80% (вместо 60-65% на синтетике)\n")
    
    # Используем реальные данные
    preference_pairs = real_preference_pairs
    logger.info(f'Using {len(preference_pairs)} REAL preference pairs from OpenAI dataset')
else:
    print("=" * 80)
    print("📝 СОЗДАЕМ СИНТЕТИЧЕСКИЕ PREFERENCE PAIRS")
    print("=" * 80)
    print("Используем улучшенную эвристику score_summary()")
    print("Для accuracy ~80% установите USE_REAL_DATASET = True в предыдущей ячейке\n")
    
    try:
        preference_pairs = []
        PAIRS_PER_PROMPT = 15  # УВЕЛИЧЕНО: 5 -> 15 для большего датасета (300 пар вместо 100)

        for idx, prompt in enumerate(prompts):
            for _ in range(PAIRS_PER_PROMPT):
                p1, p2 = random.sample(list(generated_responses.keys()), 2)
                r1 = generated_responses[p1][idx]
                r2 = generated_responses[p2][idx]
                
                # НОВАЯ ЭВРИСТИКА: многофакторная оценка вместо простого len()
                score1 = score_summary(r1, prompt)
                score2 = score_summary(r2, prompt)
                
                if score1 > score2:
                    chosen, rejected = r1, r2
                else:
                    chosen, rejected = r2, r1
                    
                preference_pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})

        logger.info(f'Created {len(preference_pairs)} synthetic preference pairs with improved scoring')
        print(f"✅ Created {len(preference_pairs)} synthetic preference pairs (improved scoring)")
    except Exception as e:
        logger.exception(f"Failed to create preference pairs: {e}")
        print(f"❌ Failed to create preference pairs: {e}")
        raise

# ===== 2. Вычисляем Reference Log Probs (π₀) - ОПТИМИЗИРОВАНО =====
print('\n' + "=" * 80)
print('🔄 ВЫЧИСЛЕНИЕ REFERENCE LOG PROBS (для KL-регуляризации)')
print("=" * 80)
logger.info(f'Computing reference log probs using {model_name}')

# ===== КРИТИЧНО: Освобождаем GPU память перед загрузкой reference model =====
print("🧹 Освобождение GPU памяти перед загрузкой reference model...")
if 'model' in locals():
    del model
if 'sft_model' in locals():
    del sft_model
torch.cuda.empty_cache()
gc.collect()
logger.info("GPU memory cleared before loading reference model")

try:
    ref_model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
    ref_model.eval()
    logger.info("Reference model loaded for log prob computation")
    print("✅ Reference model loaded")
except Exception as e:
    logger.exception(f"Failed to load reference model: {e}")
    print(f"❌ Failed to load reference model: {e}")
    raise

@torch.no_grad()
def compute_log_probs_batch(model, texts, batch_size=2):  # КРИТИЧНО УМЕНЬШЕН: 4 -> 2
    """ОПТИМИЗИРОВАННАЯ батчевая версия вычисления log probs с агрессивной очисткой памяти."""
    all_log_probs = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Log probs", leave=False):
        try:
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(
                batch_texts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True, 
                max_length=512
            ).to(model.device)
            
            logits = model(**inputs).logits
            
            # Shift for causal LM
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = inputs["input_ids"][..., 1:].contiguous()
            mask = inputs["attention_mask"][..., 1:].contiguous()
            
            # Compute log probs
            log_probs_all = torch.nn.functional.log_softmax(shift_logits, dim=-1)
            log_probs = torch.gather(log_probs_all, -1, shift_labels.unsqueeze(-1)).squeeze(-1)
            
            # Sum over sequence (masked)
            masked_log_probs = (log_probs * mask).sum(dim=-1)
            seq_log_probs = masked_log_probs / mask.sum(dim=-1).clamp(min=1)
            
            all_log_probs.extend(seq_log_probs.cpu().tolist())
            
            # Очищаем память после каждого батча
            del inputs, logits, shift_logits, shift_labels, mask, log_probs_all, log_probs, masked_log_probs, seq_log_probs
            
            # Агрессивная очистка каждые 20 батчей
            if (i // batch_size) % 20 == 0:
                torch.cuda.empty_cache()
                gc.collect()
        except Exception as e:
            logger.error(f"Error computing log probs for batch {i}: {e}")
            # Заполняем нулями для проблемных батчей
            all_log_probs.extend([0.0] * len(batch_texts))
            # Очищаем память при ошибке
            torch.cuda.empty_cache()
            gc.collect()
    
    return all_log_probs

try:
    # Собираем все тексты для батчевой обработки
    texts_chosen = [item["prompt"] + "\n" + item["chosen"] for item in preference_pairs]
    texts_rejected = [item["prompt"] + "\n" + item["rejected"] for item in preference_pairs]
    
    print(f"Computing log probs for {len(texts_chosen) * 2} texts...")
    LOG_PROB_BATCH_SIZE = 2  # КРИТИЧНО УМЕНЬШЕНО: 4 -> 2 для экономии памяти

    lp_chosen = compute_log_probs_batch(ref_model, texts_chosen, batch_size=LOG_PROB_BATCH_SIZE)
    lp_rejected = compute_log_probs_batch(ref_model, texts_rejected, batch_size=LOG_PROB_BATCH_SIZE)

    for i, item in enumerate(preference_pairs):
        item["ref_log_prob_chosen"] = lp_chosen[i]
        item["ref_log_prob_rejected"] = lp_rejected[i]

    logger.info(f'Reference log probs computed for {len(preference_pairs)} pairs')
    print(f"✅ Reference log probs computed for {len(preference_pairs)} pairs")
except Exception as e:
    logger.exception(f"Failed to compute reference log probs: {e}")
    print(f"❌ Failed to compute reference log probs: {e}")
    raise
finally:
    # Освобождаем память от reference model
    if 'ref_model' in locals():
        del ref_model
    torch.cuda.empty_cache()
    gc.collect()
    logger.info("Reference model unloaded, memory freed")

# ===== 3. Создаем Reward Model =====
print('\n' + "=" * 80)
print('🏗️  ИНИЦИАЛИЗАЦИЯ REWARD MODEL')
print("=" * 80)

# ===== КРИТИЧНО: Дополнительная очистка памяти перед созданием RM =====
print("🧹 Агрессивная очистка GPU памяти перед созданием Reward Model...")
# Удаляем все возможные остатки моделей
if 'model' in locals():
    del model
if 'sft_model' in locals():
    del sft_model
if 'ref_model' in locals():
    del ref_model
# Очищаем generated_responses для экономии памяти
if 'generated_responses' in locals():
    generated_responses.clear()
torch.cuda.empty_cache()
gc.collect()
import time
time.sleep(2)  # Даем время на очистку
torch.cuda.empty_cache()
logger.info("GPU memory aggressively cleared before RM initialization")
print(f"💾 Свободная память GPU: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

try:
    rm_model = Qwen2ForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,  # Регрессия на reward
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
    
    # ===== FIX: Устанавливаем pad_token_id для модели =====
    rm_model.config.pad_token_id = tokenizer.pad_token_id
    logger.info(f"Set rm_model.config.pad_token_id = {rm_model.config.pad_token_id}")
    
    # LoRA для RM (экономия памяти) - УМЕНЬШЕНЫ параметры
    lora_config_rm = LoraConfig(
        r=8,  # УМЕНЬШЕНО: 16 -> 8 для экономии памяти
        lora_alpha=16,  # УМЕНЬШЕНО: 32 -> 16
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS"
    )
    rm_model = get_peft_model(rm_model, lora_config_rm)
    rm_model.print_trainable_parameters()
    logger.info("Reward Model initialized with LoRA (reduced rank for memory)")
    print("✅ Reward Model initialized")
except Exception as e:
    logger.exception(f"Failed to initialize Reward Model: {e}")
    print(f"❌ Failed to initialize RM: {e}")
    raise

# ===== 4. Подготовка Dataset для Reward Model =====
class PreferenceDataset(torch.utils.data.Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        return self.pairs[idx]

def collate_fn(batch):
    """Collate function для батчинга."""
    prompts = [item["prompt"] for item in batch]
    chosen = [item["chosen"] for item in batch]
    rejected = [item["rejected"] for item in batch]
    
    # Токенизация с уменьшенной max_length
    chosen_inputs = tokenizer(
        [p + "\n" + c for p, c in zip(prompts, chosen)],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=384  # УМЕНЬШЕНО: 512 -> 384 для экономии памяти
    )
    rejected_inputs = tokenizer(
        [p + "\n" + r for p, r in zip(prompts, rejected)],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=384  # УМЕНЬШЕНО: 512 -> 384
    )
    
    return {
        "input_ids_chosen": chosen_inputs["input_ids"],
        "attention_mask_chosen": chosen_inputs["attention_mask"],
        "input_ids_rejected": rejected_inputs["input_ids"],
        "attention_mask_rejected": rejected_inputs["attention_mask"],
        "ref_log_prob_chosen": torch.tensor([item["ref_log_prob_chosen"] for item in batch]),
        "ref_log_prob_rejected": torch.tensor([item["ref_log_prob_rejected"] for item in batch]),
    }

# ===== 5. NLHF Reward Trainer =====
class NLHFRewardTrainer(Trainer):
    """
    NLHF Reward Trainer с KL-регуляризацией.
    
    Loss = -log σ(r(chosen) - r(rejected) + τ · (log π₀(chosen) - log π₀(rejected)))
    
    где:
    - r(·) - reward model output
    - π₀ - reference policy (базовая модель)
    - τ - KL regularization weight
    """
    def __init__(self, *args, tau=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.tau = tau
        logger.info(f"NLHF Reward Trainer initialized with τ={tau}")
    
    def compute_loss(self, model, inputs, return_outputs=False):
        """
        NLHF Loss: ranking loss + KL regularization.
        ОПТИМИЗИРОВАНО: обработка chosen и rejected последовательно для экономии памяти.
        """
        try:
            # КРИТИЧНО: Обрабатываем chosen и rejected последовательно, а не параллельно
            # Это экономит память, так как не нужно держать оба forward pass одновременно
            
            # Forward pass для chosen
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                r_chosen = model(
                    input_ids=inputs["input_ids_chosen"],
                    attention_mask=inputs["attention_mask_chosen"]
                ).logits.squeeze(-1)
            
            # Очищаем кэш после chosen
            torch.cuda.empty_cache()
            
            # Forward pass для rejected
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                r_rejected = model(
                    input_ids=inputs["input_ids_rejected"],
                    attention_mask=inputs["attention_mask_rejected"]
                ).logits.squeeze(-1)
            
            # KL regularization term
            kl_term = self.tau * (
                inputs["ref_log_prob_chosen"] - inputs["ref_log_prob_rejected"]
            ).to(r_chosen.device)
            
            # NLHF Loss: -log σ(Δr + τ · Δlog π₀)
            diff = r_chosen - r_rejected + kl_term
            loss = -torch.nn.functional.logsigmoid(diff).mean()
            
            return (loss, {"logits": r_chosen}) if return_outputs else loss
        except Exception as e:
            logger.exception(f"Error in compute_loss: {e}")
            raise

# ===== 6. Training Arguments =====
# Определяем размер датасета
dataset_size = len(preference_pairs)

# КРИТИЧНО: УМЕНЬШЕННЫЕ batch sizes для экономии памяти
if dataset_size <= 200:
    # Малый датасет (синтетика)
    max_steps = 150
    batch_size = 2  # КРИТИЧНО УМЕНЬШЕНО: 4 -> 2
    gradient_accumulation_steps = 8  # УВЕЛИЧЕНО: 4 -> 8 для компенсации
    print(f"📊 Малый датасет ({dataset_size} пар) → max_steps={max_steps}")
elif dataset_size <= 1000:
    # Средний датасет
    max_steps = 300
    batch_size = 3  # КРИТИЧНО УМЕНЬШЕНО: 6 -> 3
    gradient_accumulation_steps = 12  # УВЕЛИЧЕНО: 6 -> 12
    print(f"📊 Средний датасет ({dataset_size} пар) → max_steps={max_steps}")
else:
    # Большой датасет (реальный OpenAI)
    max_steps = 500
    batch_size = 2  # КРИТИЧНО УМЕНЬШЕНО: 4 -> 2
    gradient_accumulation_steps = 32  # УВЕЛИЧЕНО: 16 -> 32 для компенсации
    print(f"📊 Большой датасет ({dataset_size} пар) → max_steps={max_steps}")

train_dataset = PreferenceDataset(preference_pairs)

training_args = TrainingArguments(
    output_dir="nlhf_reward_model",
    max_steps=max_steps,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=2e-5,
    logging_steps=10,
    save_steps=100,
    bf16=True,
    remove_unused_columns=False,
    dataloader_num_workers=0,  # КРИТИЧНО: 4 -> 0 для избежания fork проблем
    report_to="none",
    gradient_checkpointing=True,  # ДОБАВЛЕНО: экономит память
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Улучшенная версия
    max_grad_norm=1.0,  # Стабилизация обучения
    optim="adamw_torch_fused"  # Более эффективный оптимизатор
)
TAU = 0.1
print("="*80)
print("🚀 НАЧАЛО ОБУЧЕНИЯ NLHF REWARD MODEL")
print("="*80)
logger.info(f"Starting NLHF Reward Model training (τ={TAU})...")

print(f"""⚙️  Конфигурация обучения:
   • Dataset: {'REAL (human feedback)' if USE_REAL_DATASET and 'real_preference_pairs' in locals() else 'SYNTHETIC (heuristic)'}
   • Dataset size: {dataset_size} pairs
   • Training steps: {max_steps}
   • Per-device batch size: {batch_size}
   • Gradient accumulation: {gradient_accumulation_steps}
   • Effective batch size: {batch_size * gradient_accumulation_steps}
   • Learning rate: {training_args.learning_rate}
   • KL regularization τ: {TAU}
   • Gradient checkpointing: ✅ (экономия памяти)
   • Sequential forward passes: ✅ (экономия памяти)
   • Max sequence length: 384 (reduced from 512)
   • LoRA rank: 8 (reduced from 16)
   • Expected accuracy: {'70-80%' if USE_REAL_DATASET else '60-70%'}
""")

# КРИТИЧНО: Еще одна очистка памяти перед началом обучения
torch.cuda.empty_cache()
gc.collect()
print(f"💾 Свободная память GPU перед обучением: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

try:
    rm_trainer = NLHFRewardTrainer(
        model=rm_model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collate_fn,
        tau=TAU
    )
    
    rm_trainer.train()
    
    logger.info("NLHF Reward Model training completed")
    print("\n" + "="*80)
    print("✅ REWARD MODEL TRAINING COMPLETE")
    print("="*80)
except Exception as e:
    logger.exception(f"NLHF Reward Model training failed: {e}")
    print(f"❌ RM training failed: {e}")
    raise

2025-11-25 22:57:55,204 INFO nlhf: Created 100 preference pairs
2025-11-25 22:57:55,207 INFO nlhf: Computing reference log probs using reference model Qwen/Qwen2.5-0.5B-Instruct


Computing reference log probs...


2025-11-25 22:58:08,279 INFO nlhf: Reference log probs computed for 100 pairs
2025-11-25 22:58:08,282 INFO nlhf: Initializing reward model with base Qwen/Qwen2.5-0.5B-Instruct


Initializing Reward Model...


2025-11-25 22:58:13,619 INFO nlhf: Starting Regularized Reward Model training...


{'loss': 2.3891, 'grad_norm': 31.5299072265625, 'learning_rate': 8e-05, 'epoch': 0.8}
{'loss': 2.236, 'grad_norm': 51.811222076416016, 'learning_rate': 6e-05, 'epoch': 1.6}
{'loss': 1.3718, 'grad_norm': 19.165895462036133, 'learning_rate': 4e-05, 'epoch': 2.4}
{'loss': 1.679, 'grad_norm': 48.670352935791016, 'learning_rate': 2e-05, 'epoch': 3.2}
{'loss': 1.2362, 'grad_norm': 35.21220016479492, 'learning_rate': 0.0, 'epoch': 4.0}
{'train_runtime': 52.6688, 'train_samples_per_second': 7.595, 'train_steps_per_second': 0.949, 'train_loss': 1.782422981262207, 'epoch': 4.0}


2025-11-25 22:59:06,559 INFO nlhf: Reward Model training completed.


Reward Model trained.


In [22]:
import os
# Install necessary libraries for visualization if not present
try:
    import matplotlib.pyplot as plt
    import pandas as pd
    import seaborn as sns
except ImportError:
    !pip install matplotlib pandas seaborn
    import matplotlib.pyplot as plt
    import pandas as pd
    import seaborn as sns

logger.info('Starting visualization. Saving plots to %s', VIS_DIR)

2025-11-25 22:59:21,516 INFO nlhf: Starting visualization. Saving plots to /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations


In [ ]:
# ===== SAMPLING: GENERATE EXAMPLES WITH REWARD MODEL =====
# Генерируем примеры ответов и оцениваем их с помощью обученной Reward Model

print("="*80)
print("[SAMPLING] GENERATING EXAMPLES WITH REWARD MODEL")
print("="*80)

import torch
import numpy as np
from tqdm import tqdm

# Проверяем наличие ref_model для генерации
if 'ref_model' not in locals() and 'ref_model' not in globals():
    print("\n[INFO] Loading base model for generation...")
    from transformers import AutoModelForCausalLM
    
    # Используем ту же модель, что и для RM, но без LoRA адаптеров
    gen_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    gen_model.eval()
    print(f"[SUCCESS] Loaded {model_name} for generation")
else:
    gen_model = ref_model
    print("[INFO] Using existing ref_model for generation")

# Тестовые промпты для демонстрации
test_prompts = [
    "Explain what is machine learning in simple terms.",
    "Write a Python function to calculate fibonacci numbers.",
    "What are the benefits of exercise?",
    "How does photosynthesis work?",
    "What is the capital of France?",
]

print(f"\n[INFO] Generating responses for {len(test_prompts)} test prompts...")
print("[INFO] Using Reward Model to score each response")

# Убеждаемся что модели в режиме eval
rm_model.eval()
gen_model.eval()

# Параметры генерации
generation_config = {
    "max_new_tokens": 128,
    "temperature": 0.9,
    "top_p": 0.95,
    "do_sample": True,
    "num_return_sequences": 3,  # Генерируем 3 варианта для каждого промпта
}

all_samples = []

with torch.no_grad():
    for prompt_idx, prompt in enumerate(test_prompts):
        print(f"\n{'='*80}")
        print(f"[PROMPT {prompt_idx+1}/{len(test_prompts)}] {prompt}")
        print(f"{'='*80}")
        
        # Генерируем несколько вариантов ответов
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(gen_model.device)
        
        outputs = gen_model.generate(
            **inputs,
            **generation_config,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        
        # Декодируем ответы
        responses = []
        for output in outputs:
            response = tokenizer.decode(output, skip_special_tokens=True)
            # Убираем промпт из ответа
            if response.startswith(prompt):
                response = response[len(prompt):].strip()
            responses.append(response)
        
        # Оцениваем каждый ответ с помощью Reward Model
        rewards = []
        for response in responses:
            # Токенизируем промпт + ответ
            full_text = prompt + "\n" + response
            reward_inputs = tokenizer(
                full_text, 
                return_tensors="pt", 
                padding=True, 
                truncation=True,
                max_length=384
            ).to(rm_model.device)
            
            # Получаем reward score
            reward_output = rm_model(**reward_inputs)
            reward_score = reward_output.logits.squeeze(-1).cpu().item()
            rewards.append(reward_score)
        
        # Сортируем по убыванию reward
        sorted_indices = np.argsort(rewards)[::-1]
        
        # Выводим результаты
        for rank, idx in enumerate(sorted_indices):
            print(f"\n[RANK {rank+1}] Reward Score: {rewards[idx]:.4f}")
            print(f"Response: {responses[idx][:200]}..." if len(responses[idx]) > 200 else f"Response: {responses[idx]}")
            
            all_samples.append({
                "prompt": prompt,
                "response": responses[idx],
                "reward": rewards[idx],
                "rank": rank + 1,
            })

print("\n" + "="*80)
print(f"[SUCCESS] Generated {len(all_samples)} samples total")
print("="*80)

# Статистика по наградам
all_rewards = [s["reward"] for s in all_samples]
print(f"\n[STATS] Reward Statistics:")
print(f"  Mean Reward: {np.mean(all_rewards):.4f}")
print(f"  Std Reward: {np.std(all_rewards):.4f}")
print(f"  Min Reward: {np.min(all_rewards):.4f}")
print(f"  Max Reward: {np.max(all_rewards):.4f}")

# Сохраняем примеры в файл
samples_file = "sample_generations_with_rewards.txt"
with open(samples_file, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("NLHF REWARD MODEL - SAMPLE GENERATIONS\n")
    f.write("="*80 + "\n\n")
    
    for i, sample in enumerate(all_samples):
        f.write(f"\n{'='*80}\n")
        f.write(f"Sample {i+1} | Reward: {sample['reward']:.4f} | Rank: {sample['rank']}\n")
        f.write(f"{'='*80}\n")
        f.write(f"Prompt: {sample['prompt']}\n\n")
        f.write(f"Response: {sample['response']}\n")

print(f"\n[FILE] Saved samples to: {samples_file}")
logger.info(f'Generated {len(all_samples)} samples, saved to {samples_file}')

# Визуализация распределения наград
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Histogram наград
ax1 = axes[0]
sns.histplot(all_rewards, bins=20, kde=True, color='skyblue', ax=ax1)
ax1.axvline(x=np.mean(all_rewards), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_rewards):.3f}')
ax1.set_title('Distribution of Reward Scores', fontsize=14, fontweight='bold')
ax1.set_xlabel('Reward Score')
ax1.set_ylabel('Frequency')
ax1.legend()
ax1.grid(True, alpha=0.3)

# График 2: Награды по промптам
ax2 = axes[1]
prompt_labels = [f"P{i+1}" for i in range(len(test_prompts))]
rewards_by_prompt = [[] for _ in range(len(test_prompts))]
for sample in all_samples:
    prompt_idx = test_prompts.index(sample["prompt"])
    rewards_by_prompt[prompt_idx].append(sample["reward"])

positions = []
values = []
for i, rewards in enumerate(rewards_by_prompt):
    positions.extend([i] * len(rewards))
    values.extend(rewards)

ax2.scatter(positions, values, alpha=0.6, s=100, c='coral')
ax2.boxplot(rewards_by_prompt, labels=prompt_labels, showfliers=False, patch_artist=True,
            boxprops=dict(facecolor='lightblue', alpha=0.5))
ax2.set_title('Reward Scores by Prompt', fontsize=14, fontweight='bold')
ax2.set_xlabel('Prompt')
ax2.set_ylabel('Reward Score')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
sample_viz_path = os.path.join(VIS_DIR, 'sample_rewards_distribution.png')
plt.savefig(sample_viz_path, bbox_inches='tight', dpi=150)
plt.show()

print(f"[FILE] Saved visualization to: {sample_viz_path}")
logger.info(f'Saved sample rewards visualization to {sample_viz_path}')

print("\n" + "="*80)
print("[COMPLETE] Sampling finished successfully!")
print("="*80)


In [ ]:
# --- Plot 1: Reward Model Training Loss ---
try:
    # Извлекаем log history из rm_trainer
    if 'rm_trainer' in locals() or 'rm_trainer' in globals():
        rm_log_history = rm_trainer.state.log_history
        logger.info(f'Extracted {len(rm_log_history)} log entries from rm_trainer')
    else:
        # Пытаемся загрузить из checkpoint
        logger.warning('rm_trainer not found in memory, attempting to load from checkpoint')
        import json
        checkpoint_dir = "nlhf_reward_model"
        checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
        if checkpoints:
            latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
            trainer_state_path = os.path.join(checkpoint_dir, latest, "trainer_state.json")
            with open(trainer_state_path, 'r') as f:
                trainer_state = json.load(f)
            rm_log_history = trainer_state['log_history']
            logger.info(f'Loaded {len(rm_log_history)} log entries from {latest}')
        else:
            raise ValueError("No rm_trainer found and no checkpoints available")
    
    # Создаем график
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    # Reward Model Loss
    rm_loss_data = [{"step": e["step"], "loss": e["loss"]} for e in rm_log_history if "loss" in e and "step" in e]
    if rm_loss_data:
        df_rm = pd.DataFrame(rm_loss_data)
        ax.plot(df_rm["step"], df_rm["loss"], 'r-', linewidth=2, marker='o', markersize=4)
        ax.set_title("NLHF Reward Model Training Loss", fontsize=16, fontweight='bold')
        ax.set_xlabel("Training Steps", fontsize=12)
        ax.set_ylabel("Loss", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.set_yscale('log')  # Логарифмическая шкала для loss
        
        # Добавляем статистику
        final_loss = df_rm["loss"].iloc[-1]
        min_loss = df_rm["loss"].min()
        ax.text(0.02, 0.98, f'Final Loss: {final_loss:.4f}\nMin Loss: {min_loss:.4f}', 
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        logger.info(f'RM Training: Final loss={final_loss:.4f}, Min loss={min_loss:.4f}')
    else:
        logger.warning("No RM loss data available for plotting")
        ax.text(0.5, 0.5, 'No training data available', ha='center', va='center', fontsize=14)

    plt.tight_layout()
    loss_path = os.path.join(VIS_DIR, 'training_loss_rm.png')
    plt.savefig(loss_path, bbox_inches='tight', dpi=150)
    plt.close()
    logger.info('Saved RM training loss plot to %s', loss_path)
    print(f"[+] Reward Model training loss plot saved to: {loss_path}")
except Exception as e:
    logger.exception(f"Failed to create training loss plot: {e}")
    print(f"[ERROR] Failed to create training loss plot: {e}")


2025-11-25 22:59:26,353 INFO nlhf: Saved training loss plot to /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/training_loss.png


✓ Training loss plot saved to: /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/training_loss.png


In [ ]:
# --- Evaluate Reward Model on Validation Set ---
print("="*80)
print("[EVAL] EVALUATING REWARD MODEL")
print("="*80)

try:
    import torch
    import numpy as np
    from tqdm import tqdm
    
    # Убеждаемся, что модель в режиме eval
    rm_model.eval()
    
    # Подготовка данных для оценки
    print(f"Evaluating on {len(preference_pairs)} preference pairs...")
    
    rewards_chosen = []
    rewards_rejected = []
    
    # Батчевая обработка для ускорения
    eval_batch_size = 4
    
    with torch.no_grad():
        for i in tqdm(range(0, len(preference_pairs), eval_batch_size), desc="Evaluating"):
            batch = preference_pairs[i:i+eval_batch_size]
            
            # Токенизация chosen
            prompts = [item["prompt"] for item in batch]
            chosen = [item["chosen"] for item in batch]
            rejected = [item["rejected"] for item in batch]
            
            chosen_inputs = tokenizer(
                [p + "\n" + c for p, c in zip(prompts, chosen)],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=384
            ).to(rm_model.device)
            
            rejected_inputs = tokenizer(
                [p + "\n" + r for p, r in zip(prompts, rejected)],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=384
            ).to(rm_model.device)
            
            # Получаем rewards
            r_chosen = rm_model(**chosen_inputs).logits.squeeze(-1).cpu().numpy()
            r_rejected = rm_model(**rejected_inputs).logits.squeeze(-1).cpu().numpy()
            
            rewards_chosen.extend(r_chosen.tolist() if r_chosen.ndim > 0 else [r_chosen.item()])
            rewards_rejected.extend(r_rejected.tolist() if r_rejected.ndim > 0 else [r_rejected.item()])
    
    # Вычисляем margins и accuracy
    rewards_chosen = np.array(rewards_chosen)
    rewards_rejected = np.array(rewards_rejected)
    margins = rewards_chosen - rewards_rejected
    accuracy = (margins > 0).mean()
    
    # Статистика
    print("\n" + "="*80)
    print("[RESULTS] REWARD MODEL EVALUATION RESULTS")
    print("="*80)
    print(f"[+] Accuracy: {accuracy:.2%}")
    print(f"[+] Mean Margin: {margins.mean():.4f} (+/-{margins.std():.4f})")
    print(f"[+] Mean Reward (Chosen): {rewards_chosen.mean():.4f}")
    print(f"[+] Mean Reward (Rejected): {rewards_rejected.mean():.4f}")
    print(f"[+] Correct Predictions: {(margins > 0).sum()} / {len(margins)}")
    print("="*80)
    
    logger.info(f'RM Evaluation: Accuracy={accuracy:.2%}, Mean margin={margins.mean():.4f}')
    
except Exception as e:
    logger.exception(f"Failed to evaluate Reward Model: {e}")
    print(f"[ERROR] Failed to evaluate RM: {e}")
    # Создаем пустые массивы для избежания ошибок в следующих ячейках
    rewards_chosen = np.array([0.0])
    rewards_rejected = np.array([0.0])
    margins = np.array([0.0])
    accuracy = 0.0
    raise


In [ ]:
# --- Plot 2: NLHF Reward Analysis (with KL regularization) ---
import numpy as np

print("Analyzing NLHF Reward Model results...")

try:
    # Create 2x2 subplot
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Plot 1: Reward Distributions
    ax1 = axes[0, 0]
    sns.histplot(rewards_chosen, color="green", label="Chosen", kde=True, alpha=0.5, ax=ax1)
    sns.histplot(rewards_rejected, color="red", label="Rejected", kde=True, alpha=0.5, ax=ax1)
    ax1.set_title(f"NLHF Reward Distributions\n(tau={TAU}, KL-regularized)", fontsize=12, fontweight='bold')
    ax1.set_xlabel("Reward Score")
    ax1.set_ylabel("Frequency")
    ax1.legend()
    ax1.axvline(x=0, color='black', linestyle='--', alpha=0.5)

    # Plot 2: Reward Margins Distribution
    ax2 = axes[0, 1]
    sns.histplot(margins, color="purple", kde=True, ax=ax2)
    ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Decision boundary')
    ax2.axvline(x=np.mean(margins), color='green', linestyle='-', linewidth=2, label=f'Mean: {np.mean(margins):.2f}')
    ax2.set_title(f"Reward Margins (Acc: {accuracy:.1%})", fontsize=12, fontweight='bold')
    ax2.set_xlabel("Margin (Chosen - Rejected)")
    ax2.set_ylabel("Frequency")
    ax2.legend()

    # Plot 3: Reference Log Prob Analysis (KL term)
    n_pairs_to_plot = min(200, len(preference_pairs))
    ref_lp_chosen = [item["ref_log_prob_chosen"] for item in preference_pairs[:n_pairs_to_plot]]
    ref_lp_rejected = [item["ref_log_prob_rejected"] for item in preference_pairs[:n_pairs_to_plot]]
    kl_terms = [TAU * (c - r) for c, r in zip(ref_lp_chosen, ref_lp_rejected)]

    ax3 = axes[1, 0]
    common_n = min(len(kl_terms), len(margins))
    if common_n > 1:
        x_vals = kl_terms[:common_n]
        y_vals = margins[:common_n]
        ax3.scatter(x_vals, y_vals, alpha=0.5, c='teal')
        ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        ax3.axvline(x=0, color='red', linestyle='--', alpha=0.5)
        ax3.set_title("KL Term vs Reward Margin\n(NLHF Core Mechanism)", fontsize=12, fontweight='bold')
        ax3.set_xlabel(f"tau * (log pi_0(chosen) - log pi_0(rejected)) [tau={TAU}]")
        ax3.set_ylabel("r(chosen) - r(rejected)")
        # Add correlation
        corr = np.corrcoef(x_vals, y_vals)[0, 1] if len(x_vals) > 1 else 0.0
        ax3.text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=ax3.transAxes, 
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    else:
        ax3.set_title("KL Term vs Reward Margin (insufficient data)", fontsize=12, fontweight='bold')
        ax3.set_xlabel(f"tau * Delta log pi_0 [tau={TAU}]")
        ax3.set_ylabel("r(chosen) - r(rejected)")
        corr = 0.0
        logger.warning("Not enough data to plot KL vs Margin")

    # Plot 4: Distribution of KL terms
    ax4 = axes[1, 1]
    if len(kl_terms) > 0:
        ax4.hist(kl_terms, bins=30, color='coral', edgecolor='white', alpha=0.7)
        ax4.axvline(x=np.mean(kl_terms), color='blue', linestyle='--', linewidth=2, label=f'Mean: {np.mean(kl_terms):.3f}')
        ax4.set_title("Distribution of KL Regularization Terms", fontsize=12, fontweight='bold')
        ax4.set_xlabel(f"tau * Delta log pi_0")
        ax4.set_ylabel("Frequency")
        ax4.legend()
    else:
        ax4.set_title("Distribution of KL Regularization Terms (empty)", fontsize=12, fontweight='bold')
        ax4.set_xlabel(f"tau * Delta log pi_0")
        ax4.set_ylabel("Frequency")

    # Вычисляем avg_margin для статистики
    avg_margin = margins.mean()

    plt.tight_layout()
    reward_path = os.path.join(VIS_DIR, 'nlhf_reward_analysis.png')
    plt.savefig(reward_path, bbox_inches='tight', dpi=150)
    plt.close()
    logger.info('Saved NLHF reward analysis plot to %s', reward_path)
    print(f"[+] NLHF Reward analysis plot saved to: {reward_path}")

    logger.info(f'NLHF RM - Accuracy: {accuracy:.2%}, Avg Margin: {avg_margin:.4f}, KL Corr: {corr:.3f}')
    print(f"\n[STATS] NLHF Statistics:")
    print(f"   RM Accuracy: {accuracy:.2%}")
    print(f"   Avg Margin: {avg_margin:.4f}")
    print(f"   KL-Reward Correlation: {corr:.3f}")
except Exception as e:
    logger.exception(f"Failed to create NLHF reward analysis plot: {e}")
    print(f"[ERROR] Failed to create NLHF reward analysis plot: {e}")
    corr = 0.0  # Default value if plot fails


Evaluating rewards on a subset of preference pairs...


2025-11-25 22:59:31,821 INFO nlhf: Saved reward distribution plot to /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/reward_distribution.png
2025-11-25 22:59:31,824 INFO nlhf: Reward Model Accuracy on subset: 42.00%


✓ Reward distribution plot saved to: /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/reward_distribution.png
Reward Model Accuracy on subset: 42.00%


In [ ]:
# ===== COMPREHENSIVE RESULTS VISUALIZATION =====
# Этот график показывает все ключевые результаты обучения NLHF Reward Model

print("="*80)
print("[RESULTS] CREATING COMPREHENSIVE RESULTS VISUALIZATION")
print("="*80)

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import seaborn as sns
    from matplotlib.gridspec import GridSpec
    
    # Создаем большую фигуру с несколькими subplot
    fig = plt.figure(figsize=(20, 12))
    gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)
    
    # ========== 1. TRAINING LOSS (большой график) ==========
    ax1 = fig.add_subplot(gs[0, :2])
    if 'rm_log_history' in locals() and rm_log_history:
        loss_data = [{"step": e["step"], "loss": e["loss"], "epoch": e.get("epoch", 0)} 
                     for e in rm_log_history if "loss" in e and "step" in e]
        if loss_data:
            df_loss = pd.DataFrame(loss_data)
            ax1.plot(df_loss["step"], df_loss["loss"], 'b-', linewidth=2.5, marker='o', markersize=4, markevery=max(1, len(df_loss)//20))
            ax1.set_title("NLHF Reward Model: Training Loss Progress", fontsize=14, fontweight='bold')
            ax1.set_xlabel("Training Steps", fontsize=12)
            ax1.set_ylabel("Loss", fontsize=12)
            ax1.grid(True, alpha=0.3)
            ax1.set_yscale('log')
            
            # Добавляем аннотации
            final_loss = df_loss["loss"].iloc[-1]
            min_loss = df_loss["loss"].min()
            max_epoch = df_loss["epoch"].max() if "epoch" in df_loss else 0
            
            textstr = f'Final Loss: {final_loss:.4f}\nMin Loss: {min_loss:.4f}\nEpochs: {max_epoch:.1f}\nSteps: {len(df_loss)}'
            ax1.text(0.02, 0.98, textstr, transform=ax1.transAxes, fontsize=11,
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
            
            # Добавляем линию минимума
            ax1.axhline(y=min_loss, color='green', linestyle='--', alpha=0.5, label=f'Min: {min_loss:.4f}')
            ax1.legend(loc='upper right')
    
    # ========== 2. ACCURACY GAUGE ==========
    ax2 = fig.add_subplot(gs[0, 2])
    if 'accuracy' in locals():
        # Создаем gauge chart для accuracy
        theta = np.linspace(0, np.pi, 100)
        r = np.ones_like(theta)
        
        # Background arc
        ax2.plot(theta, r, 'lightgray', linewidth=20, solid_capstyle='round')
        
        # Accuracy arc
        acc_theta = np.linspace(0, np.pi * accuracy, 100)
        color = 'green' if accuracy > 0.7 else 'orange' if accuracy > 0.5 else 'red'
        ax2.plot(acc_theta, np.ones_like(acc_theta), color, linewidth=20, solid_capstyle='round')
        
        # Text
        ax2.text(0, 0, f'{accuracy:.1%}', ha='center', va='center', fontsize=32, fontweight='bold')
        ax2.text(0, -0.3, 'Accuracy', ha='center', va='center', fontsize=14)
        
        ax2.set_xlim(-0.2, np.pi + 0.2)
        ax2.set_ylim(-0.5, 1.3)
        ax2.axis('off')
        ax2.set_title('Model Accuracy', fontsize=12, fontweight='bold', pad=10)
    
    # ========== 3. REWARD DISTRIBUTIONS ==========
    ax3 = fig.add_subplot(gs[1, 0])
    if 'rewards_chosen' in locals() and 'rewards_rejected' in locals():
        ax3.hist(rewards_chosen, bins=40, color='green', alpha=0.6, label='Chosen', edgecolor='white')
        ax3.hist(rewards_rejected, bins=40, color='red', alpha=0.6, label='Rejected', edgecolor='white')
        ax3.axvline(x=rewards_chosen.mean(), color='darkgreen', linestyle='--', linewidth=2)
        ax3.axvline(x=rewards_rejected.mean(), color='darkred', linestyle='--', linewidth=2)
        ax3.set_title('Reward Distributions', fontsize=12, fontweight='bold')
        ax3.set_xlabel('Reward Score')
        ax3.set_ylabel('Frequency')
        ax3.legend()
        ax3.grid(True, alpha=0.3, axis='y')
        
        # Добавляем статистику
        textstr = f'Chosen mu: {rewards_chosen.mean():.3f}\nRejected mu: {rewards_rejected.mean():.3f}\nDelta mu: {rewards_chosen.mean()-rewards_rejected.mean():.3f}'
        ax3.text(0.98, 0.98, textstr, transform=ax3.transAxes, fontsize=9,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # ========== 4. MARGIN DISTRIBUTION ==========
    ax4 = fig.add_subplot(gs[1, 1])
    if 'margins' in locals():
        colors = ['green' if m > 0 else 'red' for m in margins]
        ax4.hist(margins, bins=50, color='purple', alpha=0.7, edgecolor='white')
        ax4.axvline(x=0, color='red', linestyle='--', linewidth=2.5, label='Decision Boundary')
        ax4.axvline(x=margins.mean(), color='blue', linestyle='-', linewidth=2, label=f'Mean: {margins.mean():.3f}')
        ax4.set_title(f'Reward Margins (Acc: {accuracy:.1%})', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Margin (Chosen - Rejected)')
        ax4.set_ylabel('Frequency')
        ax4.legend()
        ax4.grid(True, alpha=0.3, axis='y')
        
        # Закрашиваем области
        ax4.axvspan(0, margins.max(), alpha=0.1, color='green', label='Correct')
        ax4.axvspan(margins.min(), 0, alpha=0.1, color='red', label='Wrong')
    
    # ========== 5. KL REGULARIZATION EFFECT ==========
    ax5 = fig.add_subplot(gs[1, 2])
    if 'preference_pairs' in locals() and len(preference_pairs) > 0:
        n_pairs = min(300, len(preference_pairs))
        kl_diffs = [TAU * (item["ref_log_prob_chosen"] - item["ref_log_prob_rejected"]) 
                   for item in preference_pairs[:n_pairs]]
        
        ax5.hist(kl_diffs, bins=40, color='coral', alpha=0.7, edgecolor='white')
        ax5.axvline(x=np.mean(kl_diffs), color='blue', linestyle='--', linewidth=2, 
                   label=f'Mean: {np.mean(kl_diffs):.4f}')
        ax5.set_title(f'KL Regularization Terms (tau={TAU})', fontsize=12, fontweight='bold')
        ax5.set_xlabel('tau * (log pi_0(chosen) - log pi_0(rejected))')
        ax5.set_ylabel('Frequency')
        ax5.legend()
        ax5.grid(True, alpha=0.3, axis='y')
    
    # ========== 6. CONFUSION MATRIX STYLE ==========
    ax6 = fig.add_subplot(gs[2, 0])
    if 'margins' in locals():
        correct = (margins > 0).sum()
        incorrect = (margins <= 0).sum()
        total = len(margins)
        
        confusion_data = np.array([[correct, incorrect]])
        im = ax6.imshow(confusion_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=total)
        
        ax6.set_xticks([0, 1])
        ax6.set_xticklabels(['Correct', 'Incorrect'])
        ax6.set_yticks([0])
        ax6.set_yticklabels(['Predictions'])
        
        # Добавляем числа
        ax6.text(0, 0, f'{correct}\n({correct/total:.1%})', ha='center', va='center', 
                fontsize=16, fontweight='bold', color='white' if correct < total/2 else 'black')
        ax6.text(1, 0, f'{incorrect}\n({incorrect/total:.1%})', ha='center', va='center', 
                fontsize=16, fontweight='bold', color='white' if incorrect > total/2 else 'black')
        
        ax6.set_title('Prediction Results', fontsize=12, fontweight='bold')
        plt.colorbar(im, ax=ax6)
    
    # ========== 7. LEARNING RATE SCHEDULE ==========
    ax7 = fig.add_subplot(gs[2, 1])
    if 'rm_log_history' in locals() and rm_log_history:
        lr_data = [{"step": e["step"], "lr": e["learning_rate"]} 
                  for e in rm_log_history if "learning_rate" in e and "step" in e]
        if lr_data:
            df_lr = pd.DataFrame(lr_data)
            ax7.plot(df_lr["step"], df_lr["lr"], 'g-', linewidth=2)
            ax7.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
            ax7.set_xlabel('Training Steps')
            ax7.set_ylabel('Learning Rate')
            ax7.grid(True, alpha=0.3)
            ax7.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    
    # ========== 8. KEY METRICS SUMMARY ==========
    ax8 = fig.add_subplot(gs[2, 2])
    ax8.axis('off')
    
    # Собираем все метрики
    summary_text = "[RESULTS] NLHF REWARD MODEL SUMMARY\n" + "="*35 + "\n\n"
    
    if 'accuracy' in locals():
        summary_text += f"[+] Accuracy: {accuracy:.2%}\n"
    if 'margins' in locals():
        summary_text += f"[+] Mean Margin: {margins.mean():.4f}\n"
        summary_text += f"[+] Margin Std: {margins.std():.4f}\n"
    if 'rewards_chosen' in locals() and 'rewards_rejected' in locals():
        summary_text += f"[+] Reward Gap: {rewards_chosen.mean()-rewards_rejected.mean():.4f}\n"
    if 'preference_pairs' in locals():
        summary_text += f"[+] Dataset Size: {len(preference_pairs)} pairs\n"
    if 'rm_log_history' in locals() and rm_log_history:
        loss_data = [e for e in rm_log_history if "loss" in e]
        if loss_data:
            summary_text += f"[+] Final Loss: {loss_data[-1]['loss']:.4f}\n"
            summary_text += f"[+] Min Loss: {min([e['loss'] for e in loss_data]):.4f}\n"
    
    summary_text += f"\n[+] KL Weight tau: {TAU}\n"
    summary_text += f"[+] Model: Qwen2.5-3B\n"
    summary_text += f"[+] LoRA rank: 8\n"
    
    ax8.text(0.1, 0.9, summary_text, transform=ax8.transAxes, fontsize=11,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    
    # ========== SAVE FIGURE ==========
    plt.suptitle('NLHF Reward Model Training Results', fontsize=18, fontweight='bold', y=0.995)
    
    results_path = os.path.join(VIS_DIR, 'comprehensive_results.png')
    plt.savefig(results_path, bbox_inches='tight', dpi=200)
    logger.info('Saved comprehensive results to %s', results_path)
    
    # Также сохраним в высоком разрешении для презентаций
    results_hd_path = os.path.join(VIS_DIR, 'comprehensive_results_HD.png')
    plt.savefig(results_hd_path, bbox_inches='tight', dpi=300)
    
    plt.show()
    
    print("\n" + "="*80)
    print("[SUCCESS] COMPREHENSIVE RESULTS VISUALIZATION CREATED")
    print("="*80)
    print(f"[FILE] Saved to: {results_path}")
    print(f"[FILE] HD version: {results_hd_path}")
    print("\n[METRICS] Key Metrics:")
    if 'accuracy' in locals():
        print(f"   * Accuracy: {accuracy:.2%}")
    if 'margins' in locals():
        print(f"   * Mean Margin: {margins.mean():.4f}")
        print(f"   * Correct Predictions: {(margins > 0).sum()} / {len(margins)}")
    
except Exception as e:
    logger.exception(f"Failed to create comprehensive visualization: {e}")
    print(f"[ERROR] Failed to create comprehensive visualization: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# --- Plot 3: Policy Diversity Analysis ---
import numpy as np

# Проверяем, существует ли переменная generated_responses
if 'generated_responses' not in locals() and 'generated_responses' not in globals():
    print("⚠️  Skipping Policy Diversity plot: generated_responses not available")
    print("   This plot requires running the policy generation step (Cell 14)")
    logger.warning("Skipping Policy Diversity plot - generated_responses not defined")
else:
    try:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        # Plot 1: Response Length Distribution by Policy
        policy_lengths = []
        for policy_id, responses in generated_responses.items():
            for r in responses:
                policy_lengths.append({"policy": f"P{policy_id}", "length": len(r)})

        if policy_lengths:
            df_lengths = pd.DataFrame(policy_lengths)
            sns.boxplot(data=df_lengths, x="policy", y="length", ax=axes[0, 0], palette="viridis")
            axes[0, 0].set_title("Response Length by Policy", fontsize=12, fontweight='bold')
            axes[0, 0].set_xlabel("Policy")
            axes[0, 0].set_ylabel("Response Length (chars)")
            axes[0, 0].tick_params(axis='x', rotation=45)
        else:
            logger.warning("No policy length data available")

        # Plot 2: Average response length per policy (bar chart)
        avg_lengths = {f"P{k}": np.mean([len(r) for r in v]) for k, v in generated_responses.items()}
        if avg_lengths:
            axes[0, 1].bar(avg_lengths.keys(), avg_lengths.values(), color='steelblue')
            axes[0, 1].axhline(y=np.mean(list(avg_lengths.values())), color='red', linestyle='--', label='Mean')
            axes[0, 1].set_title("Average Response Length per Policy", fontsize=12, fontweight='bold')
            axes[0, 1].set_xlabel("Policy")
            axes[0, 1].set_ylabel("Avg Length (chars)")
            axes[0, 1].legend()
            axes[0, 1].tick_params(axis='x', rotation=45)

        # Plot 3: Response length variance across policies (heatmap style)
        n_policies = len(generated_responses)
        if n_policies > 0:
            length_matrix = np.zeros((n_policies, n_policies))
            policy_ids = list(generated_responses.keys())
            for i, pi in enumerate(policy_ids):
                for j, pj in enumerate(policy_ids):
                    lengths_i = [len(r) for r in generated_responses[pi]]
                    lengths_j = [len(r) for r in generated_responses[pj]]
                    # Compute difference in average lengths
                    length_matrix[i, j] = abs(np.mean(lengths_i) - np.mean(lengths_j))

            im = axes[1, 0].imshow(length_matrix, cmap='YlOrRd', aspect='auto')
            axes[1, 0].set_xticks(range(n_policies))
            axes[1, 0].set_yticks(range(n_policies))
            axes[1, 0].set_xticklabels([f"P{p}" for p in policy_ids], rotation=45)
            axes[1, 0].set_yticklabels([f"P{p}" for p in policy_ids])
            axes[1, 0].set_title("Policy Length Difference Matrix", fontsize=12, fontweight='bold')
            plt.colorbar(im, ax=axes[1, 0])

        # Plot 4: Number of unique responses per policy
        unique_counts = {f"P{k}": len(set(v)) for k, v in generated_responses.items()}
        total_counts = {f"P{k}": len(v) for k, v in generated_responses.items()}
        if unique_counts:
            x = list(unique_counts.keys())
            y_unique = list(unique_counts.values())
            y_total = [total_counts[k] for k in x]

            x_pos = np.arange(len(x))
            width = 0.35
            axes[1, 1].bar(x_pos - width/2, y_total, width, label='Total', color='lightblue')
            axes[1, 1].bar(x_pos + width/2, y_unique, width, label='Unique', color='darkblue')
            axes[1, 1].set_xticks(x_pos)
            axes[1, 1].set_xticklabels(x, rotation=45)
            axes[1, 1].set_title("Response Uniqueness by Policy", fontsize=12, fontweight='bold')
            axes[1, 1].set_xlabel("Policy")
            axes[1, 1].set_ylabel("Count")
            axes[1, 1].legend()

        plt.tight_layout()
        diversity_path = os.path.join(VIS_DIR, 'policy_diversity.png')
        plt.savefig(diversity_path, bbox_inches='tight', dpi=150)
        plt.close()
        logger.info('Saved policy diversity plot to %s', diversity_path)
        print(f"✓ Policy diversity plot saved to: {diversity_path}")
    except Exception as e:
        logger.exception(f"Failed to create policy diversity plot: {e}")
        print(f"❌ Failed to create policy diversity plot: {e}")


2025-11-25 22:59:32,450 INFO nlhf: Saved policy-length boxplot to /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/policy_length_boxplot.png


✓ Policy length boxplot saved to: /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/policy_length_boxplot.png


In [ ]:
# --- Plot 4: NLHF Training Analysis ---
try:
    # Получаем rm_log_history (если еще не загружен)
    if 'rm_log_history' not in locals():
        if 'rm_trainer' in locals() or 'rm_trainer' in globals():
            rm_log_history = rm_trainer.state.log_history
        else:
            # Загружаем из checkpoint
            import json
            checkpoint_dir = "nlhf_reward_model"
            checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
            if checkpoints:
                latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
                trainer_state_path = os.path.join(checkpoint_dir, latest, "trainer_state.json")
                with open(trainer_state_path, 'r') as f:
                    trainer_state = json.load(f)
                rm_log_history = trainer_state['log_history']
            else:
                rm_log_history = []
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Plot 1: NLHF Reward Model Loss (with KL regularization)
    rm_loss_data = [{"step": e["step"], "loss": e["loss"]} for e in rm_log_history if "loss" in e and "step" in e]
    if rm_loss_data:
        df_rm = pd.DataFrame(rm_loss_data)
        axes[0, 0].plot(df_rm["step"], df_rm["loss"], 'r-', linewidth=2, marker='s', markersize=3)
        axes[0, 0].set_title(f"NLHF Reward Model Loss (τ={TAU})", fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel("Training Steps")
        axes[0, 0].set_ylabel("Loss (-log σ with KL term)")
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].set_yscale('log')
    else:
        axes[0, 0].text(0.5, 0.5, 'No RM loss data', ha='center', va='center', fontsize=14)
        axes[0, 0].set_title(f"NLHF Reward Model Loss (τ={TAU})", fontsize=12, fontweight='bold')

    # Plot 2: Learning Rate Schedule
    lr_data = [{"step": e["step"], "lr": e["learning_rate"]} for e in rm_log_history if "learning_rate" in e and "step" in e]
    if lr_data:
        df_lr = pd.DataFrame(lr_data)
        axes[0, 1].plot(df_lr["step"], df_lr["lr"], 'g-', linewidth=2)
        axes[0, 1].set_title("Learning Rate Schedule", fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel("Training Steps")
        axes[0, 1].set_ylabel("Learning Rate")
        axes[0, 1].grid(True, alpha=0.3)
        axes[0, 1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    else:
        axes[0, 1].text(0.5, 0.5, 'No LR data', ha='center', va='center', fontsize=14)
        axes[0, 1].set_title("Learning Rate Schedule", fontsize=12, fontweight='bold')

    # Plot 3: KL Divergence Analysis (Reference Model Log Probs)
    n_pairs = min(500, len(preference_pairs))
    kl_chosen = [-item["ref_log_prob_chosen"] for item in preference_pairs[:n_pairs]]
    kl_rejected = [-item["ref_log_prob_rejected"] for item in preference_pairs[:n_pairs]]

    if kl_chosen and kl_rejected:
        axes[1, 0].scatter(kl_chosen, kl_rejected, alpha=0.3, c='purple', s=10)
        min_val = min(min(kl_chosen), min(kl_rejected))
        max_val = max(max(kl_chosen), max(kl_rejected))
        axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'k--', label='y=x', linewidth=2)
        axes[1, 0].set_title("Reference Model Log Probs\n(Chosen vs Rejected)", fontsize=12, fontweight='bold')
        axes[1, 0].set_xlabel("-log π₀(chosen)")
        axes[1, 0].set_ylabel("-log π₀(rejected)")
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].text(0.5, 0.5, 'No KL data', ha='center', va='center', fontsize=14)
        axes[1, 0].set_title("Reference Model Log Probs", fontsize=12, fontweight='bold')

    # Plot 4: Preference Pair Statistics
    if preference_pairs:
        prompt_lengths = [len(item["prompt"]) for item in preference_pairs]
        chosen_lengths = [len(item["chosen"]) for item in preference_pairs]
        rejected_lengths = [len(item["rejected"]) for item in preference_pairs]
        
        axes[1, 1].hist([prompt_lengths, chosen_lengths, rejected_lengths], 
                       bins=30, label=['Prompt', 'Chosen', 'Rejected'], 
                       color=['blue', 'green', 'red'], alpha=0.6, edgecolor='white')
        axes[1, 1].set_title("Text Length Distribution in Preference Pairs", fontsize=12, fontweight='bold')
        axes[1, 1].set_xlabel("Character Length")
        axes[1, 1].set_ylabel("Frequency")
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3, axis='y')
    else:
        axes[1, 1].text(0.5, 0.5, 'No preference data', ha='center', va='center', fontsize=14)
        axes[1, 1].set_title("Text Length Distribution", fontsize=12, fontweight='bold')

    plt.tight_layout()
    analysis_path = os.path.join(VIS_DIR, 'nlhf_training_analysis.png')
    plt.savefig(analysis_path, bbox_inches='tight', dpi=150)
    plt.close()
    logger.info('Saved NLHF training analysis plot to %s', analysis_path)
    print(f"✓ NLHF Training analysis plot saved to: {analysis_path}")
except Exception as e:
    logger.exception(f"Failed to create NLHF training analysis plot: {e}")
    print(f"❌ Failed to create NLHF training analysis plot: {e}")



All visualizations saved:


2025-11-25 22:59:33,992 INFO nlhf: Visualization complete. 3 plots saved to /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations


  • /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/policy_length_boxplot.png
  • /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/reward_distribution.png
  • /home/jupyter/project/nlhf/logs/exp_20251125-223538_run/visualizations/training_loss.png


In [ ]:
# --- Plot 5: Sample Generations Comparison ---
# Эта ячейка работает только если вы запускали генерацию ответов (ячейка 14)

if 'prompts' not in locals() or 'generated_responses' not in locals():
    print("="*70)
    print("⚠️  SKIPPING SAMPLE GENERATIONS")
    print("="*70)
    print("This requires running the policy generation step (Cell 14)")
    print("Variables needed: 'prompts', 'generated_responses'")
    logger.warning("Skipping sample generations - required variables not defined")
else:
    try:
        print("\n" + "="*70)
        print("SAMPLE GENERATIONS FROM DIFFERENT POLICIES")
        print("="*70)

        # Select a few prompts to show
        sample_prompts_idx = [0, 25, 50, 75, 99] if len(prompts) >= 100 else list(range(min(5, len(prompts))))

        for prompt_idx in sample_prompts_idx:
            if prompt_idx >= len(prompts):
                break
                
            print(f"\n{'─'*70}")
            print(f"PROMPT #{prompt_idx}:")
            print(f"{'─'*70}")
            print(prompts[prompt_idx])
            print()
            
            # Baseline first
            if 'baseline' in generated_responses:
                response = generated_responses['baseline'][prompt_idx]
                print(f"POLICY_BASELINE: {response}")
            
            # Then all other policies
            for policy_id in range(N):
                if policy_id in generated_responses:
                    response = generated_responses[policy_id][prompt_idx]
                    print(f"POLICY_{policy_id}: {response}")

        # Save sample to file
        samples_path = os.path.join(VIS_DIR, 'sample_generations.txt')
        with open(samples_path, 'w', encoding='utf-8') as f:
            f.write("="*70 + "\n")
            f.write("NLHF EXPERIMENT - SAMPLE GENERATIONS\n")
            f.write(f"Model: {model_name}\n")
            f.write(f"Policies: {N} + baseline\n")
            f.write(f"Validation prompts: {len(prompts)}\n")
            f.write("="*70 + "\n\n")
            
            for prompt_idx in range(min(20, len(prompts))):
                f.write(f"\n{'─'*70}\n")
                f.write(f"PROMPT #{prompt_idx}:\n")
                f.write(f"{'─'*70}\n")
                f.write(prompts[prompt_idx] + "\n\n")
                
                # Baseline first
                if 'baseline' in generated_responses:
                    response = generated_responses['baseline'][prompt_idx]
                    f.write(f"POLICY_BASELINE: {response}\n")
                
                # Then all policies
                for policy_id in range(N):
                    if policy_id in generated_responses:
                        response = generated_responses[policy_id][prompt_idx]
                        f.write(f"POLICY_{policy_id}: {response}\n")
                
                f.write("\n")
        
        logger.info('Saved sample generations to %s', samples_path)
        print(f"\n✓ Sample generations saved to: {samples_path}")
        
    except Exception as e:
        logger.exception(f"Failed to save sample generations: {e}")
        print(f"❌ Failed to save sample generations: {e}")


In [ ]:
# --- Final Summary & Statistics ---
try:
    print("\n" + "="*70)
    print("📊 NLHF (Nash Learning from Human Feedback) EXPERIMENT SUMMARY")
    print("="*70)

    # Ensure all variables exist with defaults
    _accuracy = accuracy if 'accuracy' in dir() else 0.0
    _avg_margin = avg_margin if 'avg_margin' in dir() else 0.0
    _corr = corr if 'corr' in dir() else 0.0
    _total_responses = sum(len(v) for v in generated_responses.values()) if generated_responses else 0

    print(f"""
🔧 CONFIGURATION:
   • Base Model: {model_name}
   • SFT Dataset: {len(dataset) if 'dataset' in dir() and dataset else 'N/A'} samples
   • SFT Steps: 500
   • LoRA rank: 32, alpha: 64
   • Policies: {N}
   • Validation prompts: {len(prompts) if 'prompts' in dir() else 'N/A'}
   • Preference pairs: {len(preference_pairs) if 'preference_pairs' in dir() else 'N/A'}

🎯 NLHF-SPECIFIC PARAMETERS:
   • KL regularization τ: {TAU}
   • Reference model: π₀ = {model_name}
   • Reward Model: Trained with KL term (not pre-trained!)
   • Loss: -log σ(r(chosen) - r(rejected) + τ·Δlog π₀)

📈 RESULTS:
   • Reward Model Accuracy: {_accuracy:.2%}
   • Average Reward Margin: {_avg_margin:.4f}
   • KL-Reward Correlation: {_corr:.3f}
   • Total Generated Responses: {_total_responses}
   
📁 OUTPUT FILES:
""")

    # List all saved files
    if os.path.exists(VIS_DIR):
        for f in sorted(os.listdir(VIS_DIR)):
            fpath = os.path.join(VIS_DIR, f)
            size_kb = os.path.getsize(fpath) / 1024
            print(f"   • {f} ({size_kb:.1f} KB)")

    # Save summary to JSON
    import json
    summary = {
        "experiment_type": "NLHF",
        "base_model": model_name,
        "reference_model": model_name,
        "reward_model_type": "trained_with_kl",
        "kl_tau": TAU,
        "sft_dataset_size": len(dataset) if 'dataset' in dir() and dataset else 0,
        "sft_steps": 500,
        "rm_steps": 150,
        "lora_r": 32,
        "lora_alpha": 64,
        "n_policies": N,
        "n_validation_prompts": len(prompts) if 'prompts' in dir() else 0,
        "n_preference_pairs": len(preference_pairs) if 'preference_pairs' in dir() else 0,
        "rm_accuracy": _accuracy,
        "avg_reward_margin": _avg_margin,
        "kl_reward_correlation": _corr,
        "total_generated_responses": _total_responses,
        "nlhf_loss_formula": "-log_sigmoid(r(chosen) - r(rejected) + tau * (log_pi0_chosen - log_pi0_rejected))"
    }

    summary_path = os.path.join(EXP_DIR, 'nlhf_experiment_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"\n✅ NLHF Experiment summary saved to: {summary_path}")
    logger.info('NLHF Experiment completed. Summary saved to %s', summary_path)

    print("\n" + "="*70)
    print("✅ NLHF EXPERIMENT COMPLETED SUCCESSFULLY!")
    print("="*70)
    print("""
📚 NLHF Key Insight:
   Reward Model обучается с KL-регуляризацией к референсной модели π₀.
   Это позволяет находить Nash Equilibrium между политиками,
   избегая reward hacking и mode collapse.
   
   Loss = -log σ(r(chosen) - r(rejected) + τ · (log π₀(chosen) - log π₀(rejected)))
""")
except Exception as e:
    logger.exception(f"Failed to create final summary: {e}")
    print(f"❌ Failed to create final summary: {e}")